In [1]:
# Batch 0 / Cell 1 - Imports, project root, constants, output paths
from __future__ import annotations

import importlib
import json
import os
import platform
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from PIL import Image


def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "src" / "restoration_eval").is_dir() and (
            candidate / "tools" / "build_project_inventory.py"
        ).is_file():
            return candidate
    raise RuntimeError("Could not find thesis project root.")


PROJECT_ROOT = find_project_root()
os.chdir(PROJECT_ROOT)

SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

NOTEBOOK_ID = "24_stable_diffusion_lpips_metrics"
NOTEBOOK_LABEL = "Stable Diffusion LPIPS Metrics"
OUTPUT_ROOT = PROJECT_ROOT / "outputs" / NOTEBOOK_ID

OUTPUT_DIRS = {
    "inventory": OUTPUT_ROOT / "inventory",
    "tables": OUTPUT_ROOT / "tables",
    "metrics": OUTPUT_ROOT / "metrics",
    "analysis": OUTPUT_ROOT / "analysis",
    "figures": OUTPUT_ROOT / "figures",
    "validation": OUTPUT_ROOT / "validation",
    "manifests": OUTPUT_ROOT / "manifests",
    "reports": OUTPUT_ROOT / "reports",
}

for output_dir in OUTPUT_DIRS.values():
    output_dir.mkdir(parents=True, exist_ok=True)

INVENTORY_SCRIPT_PATH = PROJECT_ROOT / "tools" / "build_project_inventory.py"
GLOBAL_INVENTORY_DIR = PROJECT_ROOT / "outputs" / "inventory"
GLOBAL_INVENTORY_PATH = GLOBAL_INVENTORY_DIR / "project_file_inventory.csv"

BATCH0_INVENTORY_SNAPSHOT_PATH = OUTPUT_DIRS["inventory"] / "batch0_project_inventory_snapshot.csv"
BATCH0_VALIDATION_PATH = OUTPUT_DIRS["validation"] / "batch0_validation.csv"
STAGE_MANIFEST_PATH = OUTPUT_DIRS["manifests"] / "stable_diffusion_lpips_metrics_stage_manifest.json"

BATCH1_INPUT_CASES_PATH = OUTPUT_DIRS["tables"] / "stable_diffusion_lpips_input_cases.csv"
BATCH1_VALIDATION_PATH = OUTPUT_DIRS["validation"] / "batch1_input_validation.csv"

BATCH2_SMOKE_METRICS_PATH = OUTPUT_DIRS["validation"] / "stable_diffusion_lpips_smoke.csv"
BATCH2_VALIDATION_PATH = OUTPUT_DIRS["validation"] / "batch2_smoke_validation.csv"
BATCH3_METRICS_PATH = OUTPUT_DIRS["metrics"] / "stable_diffusion_lpips_metrics.csv"
BATCH3_VALIDATION_PATH = OUTPUT_DIRS["validation"] / "stable_diffusion_lpips_metrics_validation.csv"
BATCH4_SUMMARY_PATH = OUTPUT_DIRS["analysis"] / "stable_diffusion_lpips_summary.csv"
BATCH4_SELECTED_CASES_PATH = OUTPUT_DIRS["analysis"] / "stable_diffusion_lpips_selected_cases.csv"
BATCH4_VALIDATION_PATH = OUTPUT_DIRS["validation"] / "batch4_analysis_validation.csv"
BATCH5_FIGURE_MANIFEST_PATH = OUTPUT_DIRS["figures"] / "stable_diffusion_lpips_figure_manifest.csv"
BATCH6_ARTIFACT_INDEX_PATH = OUTPUT_DIRS["manifests"] / "stable_diffusion_lpips_metrics_artifact_index.csv"
BATCH6_HANDOFF_MANIFEST_PATH = OUTPUT_DIRS["manifests"] / "stable_diffusion_lpips_metrics_handoff_manifest.json"
BATCH6_FINAL_VALIDATION_PATH = OUTPUT_DIRS["validation"] / "stable_diffusion_lpips_metrics_final_validation.csv"

UPSTREAM_INPUT_CASES_PATH = (
    PROJECT_ROOT
    / "outputs"
    / "22_stable_diffusion_classical_metrics"
    / "tables"
    / "stable_diffusion_classical_metric_input_cases.csv"
)

LPIPS_NET = "alex"
LPIPS_INPUT_SIZE = 256
MASK_BBOX_MARGIN = 8
MASK_BINARY_THRESHOLD = 127
TARGET_SIZE = 768

EXPECTED_CANDIDATE_ROWS = 945
EXPECTED_ZERO_CONTROL_ROWS = 94


def utc_now_iso() -> str:
    return datetime.now(timezone.utc).isoformat()


def rel(path: str | Path) -> str:
    path = Path(path)
    try:
        return path.resolve().relative_to(PROJECT_ROOT).as_posix()
    except ValueError:
        return path.as_posix()


def resolve_project_path(path_value: str | Path) -> Path:
    path = Path(str(path_value))
    return path if path.is_absolute() else PROJECT_ROOT / path


def bool_series(series: pd.Series) -> pd.Series:
    if series.dtype == bool:
        return series.fillna(False)
    return series.astype(str).str.strip().str.lower().isin({"true", "1", "yes", "y"})


def validation_row(check_name: str, observed, expected, passed: bool, failure_message: str = "") -> dict[str, Any]:
    return {
        "check_name": check_name,
        "observed": observed,
        "expected": expected,
        "passed": bool(passed),
        "failure_message": "" if passed else failure_message,
    }


def to_json_safe(value: Any) -> Any:
    if isinstance(value, dict):
        return {str(key): to_json_safe(item) for key, item in value.items()}
    if isinstance(value, list):
        return [to_json_safe(item) for item in value]
    if isinstance(value, tuple):
        return [to_json_safe(item) for item in value]
    if isinstance(value, Path):
        return rel(value)
    if isinstance(value, np.integer):
        return int(value)
    if isinstance(value, np.floating):
        return None if np.isnan(value) else float(value)
    if isinstance(value, np.bool_):
        return bool(value)
    try:
        if pd.isna(value):
            return None
    except (TypeError, ValueError):
        pass
    return value


def read_csv_columns(path: Path) -> list[str]:
    if not path.is_file():
        return []
    return pd.read_csv(path, nrows=0).columns.tolist()


print(f"Project root: {PROJECT_ROOT}")
print(f"Notebook output root: {rel(OUTPUT_ROOT)}")
print(f"Python: {sys.version.split()[0]} | platform: {platform.platform()}")

Project root: D:\Masters\FH\Thesis\painting-restoration-eval
Notebook output root: outputs/24_stable_diffusion_lpips_metrics
Python: 3.12.6 | platform: Windows-11-10.0.26200-SP0


In [2]:
# Batch 0 / Cell 2 - Refresh project inventory and save Notebook 24 snapshot
if not INVENTORY_SCRIPT_PATH.is_file():
    raise FileNotFoundError(f"Inventory script not found: {INVENTORY_SCRIPT_PATH}")

GLOBAL_INVENTORY_DIR.mkdir(parents=True, exist_ok=True)

inventory_result = subprocess.run(
    [
        sys.executable,
        str(INVENTORY_SCRIPT_PATH),
        "--root",
        str(PROJECT_ROOT),
        "--out-dir",
        str(GLOBAL_INVENTORY_DIR),
    ],
    cwd=PROJECT_ROOT,
    capture_output=True,
    text=True,
    check=False,
)

print(inventory_result.stdout.strip())

if inventory_result.returncode != 0:
    print(inventory_result.stderr)
    raise RuntimeError("Project inventory refresh failed.")

if not GLOBAL_INVENTORY_PATH.is_file():
    raise FileNotFoundError(f"Inventory CSV was not created: {GLOBAL_INVENTORY_PATH}")

inventory_df = pd.read_csv(GLOBAL_INVENTORY_PATH)
inventory_df.to_csv(BATCH0_INVENTORY_SNAPSHOT_PATH, index=False)

print(f"Inventory rows: {len(inventory_df):,}")
print(f"Saved: {rel(BATCH0_INVENTORY_SNAPSHOT_PATH)}")

display(
    inventory_df.loc[
        inventory_df["relative_path"].astype(str).str.contains(
            "22_stable_diffusion|23_stable_diffusion|24_stable_diffusion|metrics_lpips",
            case=False,
            na=False,
        ),
        [
            column
            for column in [
                "relative_path",
                "file_kind",
                "size_bytes",
                "csv_row_count",
                "csv_column_count",
            ]
            if column in inventory_df.columns
        ],
    ].head(80)
)

Saved inventory: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\inventory\project_file_inventory.csv
Saved summary:   D:\Masters\FH\Thesis\painting-restoration-eval\outputs\inventory\project_file_inventory_summary.csv
Files indexed:   10800
Inventory rows: 10,800
Saved: outputs/24_stable_diffusion_lpips_metrics/inventory/batch0_project_inventory_snapshot.csv


,relative_path,file_kind,size_bytes,csv_row_count,csv_column_count
2854,notebooks/22_stable_diffusion_classical_metric...,notebook,545091,NaN,NaN
2856,notebooks/23_stable_diffusion_difference_maps....,notebook,7713820,NaN,NaN
2858,notebooks/24_stable_diffusion_lpips_metrics.ipynb,notebook,617,NaN,NaN
3473,outputs/22_stable_diffusion_classical_metrics/...,metric_csv,15579,11.0,65.0
3474,outputs/22_stable_diffusion_classical_metrics/...,metric_csv,25706,135.0,24.0
...,...,...,...,...,...
3545,outputs/23_stable_diffusion_difference_maps/ma...,image,656194,NaN,NaN
3546,outputs/23_stable_diffusion_difference_maps/ma...,image,656194,NaN,NaN
3547,outputs/23_stable_diffusion_difference_maps/ma...,image,656194,NaN,NaN
3548,outputs/23_stable_diffusion_difference_maps/ma...,image,656194,NaN,NaN


In [3]:
# Batch 0 / Cell 3 - Import and verify LPIPS helper API
metrics_lpips = importlib.import_module("restoration_eval.metrics_lpips")
metrics_lpips = importlib.reload(metrics_lpips)

REQUIRED_HELPER_ATTRIBUTES = [
    "LPIPS_METRIC_SCHEMA_VERSION",
    "LPIPS_EVALUATION_REGIONS",
    "validate_lpips_runtime_dependencies",
    "get_device",
    "load_lpips_model",
    "load_mask_bool",
    "get_content_box_from_row",
    "compute_lpips_metrics_for_restorations",
    "validate_lpips_metrics",
    "summarize_lpips_metrics",
    "rank_lpips_cases",
]

missing_helper_attributes = [
    attribute for attribute in REQUIRED_HELPER_ATTRIBUTES
    if not hasattr(metrics_lpips, attribute)
]

runtime_dependencies_df = metrics_lpips.validate_lpips_runtime_dependencies()
runtime_dependencies_passed = bool_series(runtime_dependencies_df["passed"]).all()

region_policy_valid = tuple(metrics_lpips.LPIPS_EVALUATION_REGIONS) == (
    "full_image",
    "content_region",
    "mask_bbox_crop",
)

print("LPIPS helper schema version:", metrics_lpips.LPIPS_METRIC_SCHEMA_VERSION)
print("LPIPS evaluation regions:", metrics_lpips.LPIPS_EVALUATION_REGIONS)
print("Missing helper attributes:", missing_helper_attributes)

display(runtime_dependencies_df)

if missing_helper_attributes:
    raise RuntimeError(f"LPIPS helper API is incomplete: {missing_helper_attributes}")

if not region_policy_valid:
    raise RuntimeError("LPIPS helper region policy does not match Notebook 24 contract.")

if not runtime_dependencies_passed:
    display(runtime_dependencies_df.loc[~bool_series(runtime_dependencies_df["passed"])])
    raise RuntimeError("LPIPS runtime dependencies are missing.")

LPIPS helper schema version: 2.1.0
LPIPS evaluation regions: ('full_image', 'content_region', 'mask_bbox_crop')
Missing helper attributes: []


,component,module,version,required,installed,passed
0,torch,torch,2.5.1+cu121,True,True,True
1,lpips,lpips,0.1.4,True,True,True
2,Pillow,PIL,11.0.0,True,True,True


In [4]:
# Batch 0 / Cell 4 - Declare upstream inputs and output contract
UPSTREAM_ARTIFACTS = {
    "notebook22_metric_input_cases": {
        "path": UPSTREAM_INPUT_CASES_PATH,
        "required": True,
        "artifact_role": "primary_input",
        "expected_rows": EXPECTED_CANDIDATE_ROWS,
    },
    "notebook22_final_validation": {
        "path": PROJECT_ROOT
        / "outputs"
        / "22_stable_diffusion_classical_metrics"
        / "validation"
        / "stable_diffusion_classical_metrics_final_validation.csv",
        "required": True,
        "artifact_role": "upstream_completion_marker",
        "expected_rows": None,
    },
    "notebook22_handoff_manifest": {
        "path": PROJECT_ROOT
        / "outputs"
        / "22_stable_diffusion_classical_metrics"
        / "manifests"
        / "stable_diffusion_classical_metrics_handoff_manifest.json",
        "required": True,
        "artifact_role": "upstream_handoff_manifest",
        "expected_rows": None,
    },
    "notebook23_final_validation": {
        "path": PROJECT_ROOT
        / "outputs"
        / "23_stable_diffusion_difference_maps"
        / "validation"
        / "stable_diffusion_difference_maps_final_validation.csv",
        "required": True,
        "artifact_role": "sequence_completion_marker",
        "expected_rows": None,
    },
    "notebook23_handoff_manifest": {
        "path": PROJECT_ROOT
        / "outputs"
        / "23_stable_diffusion_difference_maps"
        / "manifests"
        / "stable_diffusion_difference_maps_handoff_manifest.json",
        "required": True,
        "artifact_role": "sequence_handoff_manifest",
        "expected_rows": None,
    },
}

OUTPUT_CONTRACT = {
    "batch0_inventory_snapshot": rel(BATCH0_INVENTORY_SNAPSHOT_PATH),
    "batch0_validation": rel(BATCH0_VALIDATION_PATH),
    "stage_manifest": rel(STAGE_MANIFEST_PATH),
    "lpips_input_cases": rel(BATCH1_INPUT_CASES_PATH),
    "batch1_input_validation": rel(BATCH1_VALIDATION_PATH),
    "smoke_metrics": rel(BATCH2_SMOKE_METRICS_PATH),
    "batch2_smoke_validation": rel(BATCH2_VALIDATION_PATH),
    "lpips_metrics": rel(BATCH3_METRICS_PATH),
    "lpips_metrics_validation": rel(BATCH3_VALIDATION_PATH),
    "lpips_summary": rel(BATCH4_SUMMARY_PATH),
    "selected_cases": rel(BATCH4_SELECTED_CASES_PATH),
    "batch4_analysis_validation": rel(BATCH4_VALIDATION_PATH),
    "figure_manifest": rel(BATCH5_FIGURE_MANIFEST_PATH),
    "artifact_index": rel(BATCH6_ARTIFACT_INDEX_PATH),
    "handoff_manifest": rel(BATCH6_HANDOFF_MANIFEST_PATH),
    "final_validation": rel(BATCH6_FINAL_VALIDATION_PATH),
}

display(
    pd.DataFrame(
        [
            {
                "artifact_key": key,
                "artifact_role": value["artifact_role"],
                "required": value["required"],
                "expected_rows": value["expected_rows"],
                "path": rel(value["path"]),
            }
            for key, value in UPSTREAM_ARTIFACTS.items()
        ]
    )
)

display(pd.DataFrame([{"artifact_key": key, "path": value} for key, value in OUTPUT_CONTRACT.items()]))

,artifact_key,artifact_role,required,expected_rows,path
0,notebook22_metric_input_cases,primary_input,True,945.0,outputs/22_stable_diffusion_classical_metrics/...
1,notebook22_final_validation,upstream_completion_marker,True,NaN,outputs/22_stable_diffusion_classical_metrics/...
2,notebook22_handoff_manifest,upstream_handoff_manifest,True,NaN,outputs/22_stable_diffusion_classical_metrics/...
3,notebook23_final_validation,sequence_completion_marker,True,NaN,outputs/23_stable_diffusion_difference_maps/va...
4,notebook23_handoff_manifest,sequence_handoff_manifest,True,NaN,outputs/23_stable_diffusion_difference_maps/ma...


,artifact_key,path
0,batch0_inventory_snapshot,outputs/24_stable_diffusion_lpips_metrics/inve...
1,batch0_validation,outputs/24_stable_diffusion_lpips_metrics/vali...
2,stage_manifest,outputs/24_stable_diffusion_lpips_metrics/mani...
3,lpips_input_cases,outputs/24_stable_diffusion_lpips_metrics/tabl...
4,batch1_input_validation,outputs/24_stable_diffusion_lpips_metrics/vali...
5,smoke_metrics,outputs/24_stable_diffusion_lpips_metrics/vali...
6,batch2_smoke_validation,outputs/24_stable_diffusion_lpips_metrics/vali...
7,lpips_metrics,outputs/24_stable_diffusion_lpips_metrics/metr...
8,lpips_metrics_validation,outputs/24_stable_diffusion_lpips_metrics/vali...
9,lpips_summary,outputs/24_stable_diffusion_lpips_metrics/anal...


In [5]:
# Batch 0 / Cell 5 - Upstream scan and primary-input header validation
def inventory_contains_path(path: Path) -> bool:
    if "relative_path" not in inventory_df.columns:
        return False
    return rel(path) in set(inventory_df["relative_path"].astype(str))


def validation_file_passed(path: Path) -> bool | None:
    if not path.is_file() or path.suffix.lower() != ".csv":
        return None
    validation_df = pd.read_csv(path)
    if validation_df.empty or "passed" not in validation_df.columns:
        return None
    return bool(bool_series(validation_df["passed"]).all())


upstream_rows = []

for artifact_key, artifact in UPSTREAM_ARTIFACTS.items():
    artifact_path = artifact["path"]
    exists = artifact_path.is_file()
    row_count = None
    column_count = None
    columns = []
    validation_passed = validation_file_passed(artifact_path)

    if exists and artifact_path.suffix.lower() == ".csv":
        artifact_df = pd.read_csv(artifact_path)
        row_count = int(len(artifact_df))
        column_count = int(len(artifact_df.columns))
        columns = artifact_df.columns.tolist()

    upstream_rows.append(
        {
            "artifact_key": artifact_key,
            "artifact_role": artifact["artifact_role"],
            "required": bool(artifact["required"]),
            "path": rel(artifact_path),
            "exists": bool(exists),
            "in_inventory": bool(inventory_contains_path(artifact_path)),
            "row_count": row_count,
            "expected_rows": artifact["expected_rows"],
            "row_count_matches": True if artifact["expected_rows"] is None else row_count == artifact["expected_rows"],
            "column_count": column_count,
            "validation_passed": validation_passed,
            "columns": " | ".join(columns),
        }
    )

upstream_scan_df = pd.DataFrame(upstream_rows)

display(
    upstream_scan_df[
        [
            "artifact_key",
            "artifact_role",
            "exists",
            "in_inventory",
            "row_count",
            "expected_rows",
            "row_count_matches",
            "validation_passed",
        ]
    ]
)

primary_input_columns = read_csv_columns(UPSTREAM_INPUT_CASES_PATH)

REQUIRED_INPUT_COLUMNS = [
    "metric_case_id",
    "dataset_name",
    "case_id",
    "restoration_case_id",
    "candidate_id",
    "painting_id",
    "category",
    "title",
    "mask_id",
    "mask_type",
    "model_name",
    "clean_path",
    "damaged_path",
    "restored_path",
    "mask_path",
    "content_x_min",
    "content_y_min",
    "content_x_max",
    "content_y_max",
]

REQUIRED_SEED_COLUMNS = [
    "candidate_index",
    "candidate_seed",
    "prompt_policy_id",
    "prompt_variant_id",
    "prompt_template_name",
    "prompt_variant_family",
]

missing_required_input_columns = [
    column for column in REQUIRED_INPUT_COLUMNS
    if column not in primary_input_columns
]

missing_seed_columns = [
    column for column in REQUIRED_SEED_COLUMNS
    if column not in primary_input_columns
]

print("Primary input:", rel(UPSTREAM_INPUT_CASES_PATH))
print("Primary input column count:", len(primary_input_columns))
print("Missing required input columns:", missing_required_input_columns)
print("Missing seed/prompt columns:", missing_seed_columns)

,artifact_key,artifact_role,exists,in_inventory,row_count,expected_rows,row_count_matches,validation_passed
0,notebook22_metric_input_cases,primary_input,True,True,945.0,945.0,True,None
1,notebook22_final_validation,upstream_completion_marker,True,True,22.0,NaN,True,True
2,notebook22_handoff_manifest,upstream_handoff_manifest,True,True,NaN,NaN,True,None
3,notebook23_final_validation,sequence_completion_marker,True,True,19.0,NaN,True,True
4,notebook23_handoff_manifest,sequence_handoff_manifest,True,True,NaN,NaN,True,None


Primary input: outputs/22_stable_diffusion_classical_metrics/tables/stable_diffusion_classical_metric_input_cases.csv
Primary input column count: 278
Missing required input columns: []
Missing seed/prompt columns: []


In [6]:
# Batch 0 / Cell 6 - Write Batch 0 validation and stage manifest
missing_required_artifacts = upstream_scan_df.loc[
    upstream_scan_df["required"].astype(bool) & ~upstream_scan_df["exists"].astype(bool),
    "artifact_key",
].tolist()

required_missing_from_inventory = upstream_scan_df.loc[
    upstream_scan_df["required"].astype(bool) & ~upstream_scan_df["in_inventory"].astype(bool),
    "artifact_key",
].tolist()

row_count_mismatches = upstream_scan_df.loc[
    upstream_scan_df["expected_rows"].notna()
    & ~upstream_scan_df["row_count_matches"].astype(bool),
    "artifact_key",
].tolist()

failed_validations = upstream_scan_df.loc[
    upstream_scan_df["validation_passed"].eq(False),
    "artifact_key",
].tolist()

unknown_completion_validations = upstream_scan_df.loc[
    upstream_scan_df["artifact_role"].astype(str).str.contains("completion_marker")
    & upstream_scan_df["validation_passed"].isna(),
    "artifact_key",
].tolist()

batch0_validation_df = pd.DataFrame(
    [
        validation_row(
            "inventory_refreshed",
            rel(GLOBAL_INVENTORY_PATH),
            "file exists",
            GLOBAL_INVENTORY_PATH.is_file(),
            "Project inventory CSV was not refreshed.",
        ),
        validation_row(
            "inventory_snapshot_written",
            rel(BATCH0_INVENTORY_SNAPSHOT_PATH),
            "file exists",
            BATCH0_INVENTORY_SNAPSHOT_PATH.is_file(),
            "Notebook 24 inventory snapshot was not written.",
        ),
        validation_row(
            "helper_api_complete",
            missing_helper_attributes,
            [],
            len(missing_helper_attributes) == 0,
            "LPIPS helper is missing required symbols.",
        ),
        validation_row(
            "helper_region_policy",
            tuple(metrics_lpips.LPIPS_EVALUATION_REGIONS),
            ("full_image", "content_region", "mask_bbox_crop"),
            region_policy_valid,
            "LPIPS region policy is wrong.",
        ),
        validation_row(
            "runtime_dependencies_available",
            runtime_dependencies_df.loc[~bool_series(runtime_dependencies_df["passed"]), "component"].tolist(),
            [],
            runtime_dependencies_passed,
            "LPIPS runtime dependency missing.",
        ),
        validation_row(
            "required_upstream_artifacts_exist",
            missing_required_artifacts,
            [],
            len(missing_required_artifacts) == 0,
            "Required upstream artifact missing.",
        ),
        validation_row(
            "required_upstream_artifacts_in_inventory",
            required_missing_from_inventory,
            [],
            len(required_missing_from_inventory) == 0,
            "Required upstream artifact missing from refreshed inventory.",
        ),
        validation_row(
            "upstream_row_counts_match",
            row_count_mismatches,
            [],
            len(row_count_mismatches) == 0,
            "Upstream row count mismatch.",
        ),
        validation_row(
            "upstream_validation_files_passed",
            failed_validations,
            [],
            len(failed_validations) == 0,
            "Upstream validation failed.",
        ),
        validation_row(
            "upstream_completion_markers_readable",
            unknown_completion_validations,
            [],
            len(unknown_completion_validations) == 0,
            "Completion marker missing/unreadable or lacks passed column.",
        ),
        validation_row(
            "primary_input_required_columns_present",
            missing_required_input_columns,
            [],
            len(missing_required_input_columns) == 0,
            "Notebook 22 metric input cases are missing required LPIPS columns.",
        ),
        validation_row(
            "seed_prompt_columns_present",
            missing_seed_columns,
            [],
            len(missing_seed_columns) == 0,
            "Notebook 22 metric input cases are missing seed/prompt identity columns.",
        ),
    ]
)

batch0_validation_df.to_csv(BATCH0_VALIDATION_PATH, index=False)

stage_manifest = {
    "schema_version": "stable_diffusion_lpips_metrics_stage_manifest_v1",
    "notebook_id": NOTEBOOK_ID,
    "notebook_label": NOTEBOOK_LABEL,
    "stage": "batch0_setup_inventory_contract",
    "stage_status": "passed" if bool_series(batch0_validation_df["passed"]).all() else "failed",
    "updated_at_utc": utc_now_iso(),
    "output_root": rel(OUTPUT_ROOT),
    "primary_input": rel(UPSTREAM_INPUT_CASES_PATH),
    "lpips_contract": {
        "lpips_net": LPIPS_NET,
        "lpips_input_size": LPIPS_INPUT_SIZE,
        "mask_bbox_margin": MASK_BBOX_MARGIN,
        "mask_binary_threshold": MASK_BINARY_THRESHOLD,
        "target_size": TARGET_SIZE,
        "evaluation_regions": list(metrics_lpips.LPIPS_EVALUATION_REGIONS),
        "expected_candidate_rows": EXPECTED_CANDIDATE_ROWS,
        "expected_zero_control_rows": EXPECTED_ZERO_CONTROL_ROWS,
    },
    "upstream_artifacts": {
        key: {
            "path": rel(value["path"]),
            "required": bool(value["required"]),
            "artifact_role": value["artifact_role"],
            "expected_rows": value["expected_rows"],
        }
        for key, value in UPSTREAM_ARTIFACTS.items()
    },
    "outputs": OUTPUT_CONTRACT,
    "batch0": {
        "status": "passed" if bool_series(batch0_validation_df["passed"]).all() else "failed",
        "validation": rel(BATCH0_VALIDATION_PATH),
        "inventory_snapshot": rel(BATCH0_INVENTORY_SNAPSHOT_PATH),
        "checks_passed": int(bool_series(batch0_validation_df["passed"]).sum()),
        "checks_total": int(len(batch0_validation_df)),
    },
}

STAGE_MANIFEST_PATH.write_text(json.dumps(to_json_safe(stage_manifest), indent=2), encoding="utf-8")

print(f"Saved: {rel(BATCH0_VALIDATION_PATH)}")
print(f"Updated: {rel(STAGE_MANIFEST_PATH)}")
print(f"Batch 0 checks passed: {int(bool_series(batch0_validation_df['passed']).sum())} / {len(batch0_validation_df)}")

display(batch0_validation_df)

if not bool_series(batch0_validation_df["passed"]).all():
    display(batch0_validation_df.loc[~bool_series(batch0_validation_df["passed"]), ["check_name", "failure_message"]])
    raise RuntimeError("Batch 0 validation failed.")

Saved: outputs/24_stable_diffusion_lpips_metrics/validation/batch0_validation.csv
Updated: outputs/24_stable_diffusion_lpips_metrics/manifests/stable_diffusion_lpips_metrics_stage_manifest.json
Batch 0 checks passed: 12 / 12


,check_name,observed,expected,passed,failure_message
0,inventory_refreshed,outputs/inventory/project_file_inventory.csv,file exists,True,
1,inventory_snapshot_written,outputs/24_stable_diffusion_lpips_metrics/inve...,file exists,True,
2,helper_api_complete,[],[],True,
3,helper_region_policy,"(full_image, content_region, mask_bbox_crop)","(full_image, content_region, mask_bbox_crop)",True,
4,runtime_dependencies_available,[],[],True,
5,required_upstream_artifacts_exist,[],[],True,
6,required_upstream_artifacts_in_inventory,[],[],True,
7,upstream_row_counts_match,[],[],True,
8,upstream_validation_files_passed,[],[],True,
9,upstream_completion_markers_readable,[],[],True,


In [7]:
# Batch 1 / Cell 7 - Load Notebook 22 standardized metric input cases
if not bool_series(pd.read_csv(BATCH0_VALIDATION_PATH)["passed"]).all():
    raise RuntimeError("Batch 0 did not pass. Do not continue to Batch 1.")

source_cases_df = pd.read_csv(UPSTREAM_INPUT_CASES_PATH)

candidate_manifest_df = source_cases_df.copy()

print(f"Loaded: {rel(UPSTREAM_INPUT_CASES_PATH)}")
print(f"Rows: {len(candidate_manifest_df):,}")
print(f"Columns: {len(candidate_manifest_df.columns):,}")

display(
    candidate_manifest_df[
        [
            "metric_case_id",
            "candidate_id",
            "case_id",
            "mask_type",
            "clean_path",
            "damaged_path",
            "restored_path",
            "mask_path",
            "content_x_min",
            "content_y_min",
            "content_x_max",
            "content_y_max",
        ]
    ].head(10)
)

Loaded: outputs/22_stable_diffusion_classical_metrics/tables/stable_diffusion_classical_metric_input_cases.csv
Rows: 945
Columns: 278


,metric_case_id,candidate_id,case_id,mask_type,clean_path,damaged_path,restored_path,mask_path,content_x_min,content_y_min,content_x_max,content_y_max
0,sd__can__p001__loss_large__p00_generic__s2026_...,sd__can__p001__loss_large__p00_generic__s2026_...,p001_loss_large,loss_large,data/processed/clean/p001_clean.png,data/processed/masked/p001_loss_large_damaged.png,data/processed/restored/stable_diffusion/canon...,data/processed/masks/p001_loss_large_mask.png,52.0,0.0,715.0,768.0
1,sd__can__p001__loss_small__p00_generic__s2026_...,sd__can__p001__loss_small__p00_generic__s2026_...,p001_loss_small,loss_small,data/processed/clean/p001_clean.png,data/processed/masked/p001_loss_small_damaged.png,data/processed/restored/stable_diffusion/canon...,data/processed/masks/p001_loss_small_mask.png,52.0,0.0,715.0,768.0
2,sd__can__p001__mixed_damage__p00_generic__s202...,sd__can__p001__mixed_damage__p00_generic__s202...,p001_mixed_damage,mixed_damage,data/processed/clean/p001_clean.png,data/processed/masked/p001_mixed_damage_damage...,data/processed/restored/stable_diffusion/canon...,data/processed/masks/p001_mixed_damage_mask.png,52.0,0.0,715.0,768.0
3,sd__can__p001__scratch_thin__p00_generic__s202...,sd__can__p001__scratch_thin__p00_generic__s202...,p001_scratch_thin,scratch_thin,data/processed/clean/p001_clean.png,data/processed/masked/p001_scratch_thin_damage...,data/processed/restored/stable_diffusion/canon...,data/processed/masks/p001_scratch_thin_mask.png,52.0,0.0,715.0,768.0
4,sd__can__p001__zero__p00_generic__s2026__3be10...,sd__can__p001__zero__p00_generic__s2026__3be10...,p001_zero_control,zero_control,data/processed/clean/p001_clean.png,data/processed/masked/p001_zero_control_damage...,data/processed/restored/stable_diffusion/canon...,data/processed/masks/p001_zero_control_mask.png,52.0,0.0,715.0,768.0
5,sd__can__p002__loss_large__p00_generic__s2026_...,sd__can__p002__loss_large__p00_generic__s2026_...,p002_loss_large,loss_large,data/processed/clean/p002_clean.png,data/processed/masked/p002_loss_large_damaged.png,data/processed/restored/stable_diffusion/canon...,data/processed/masks/p002_loss_large_mask.png,159.0,0.0,608.0,768.0
6,sd__can__p002__loss_small__p00_generic__s2026_...,sd__can__p002__loss_small__p00_generic__s2026_...,p002_loss_small,loss_small,data/processed/clean/p002_clean.png,data/processed/masked/p002_loss_small_damaged.png,data/processed/restored/stable_diffusion/canon...,data/processed/masks/p002_loss_small_mask.png,159.0,0.0,608.0,768.0
7,sd__can__p002__mixed_damage__p00_generic__s202...,sd__can__p002__mixed_damage__p00_generic__s202...,p002_mixed_damage,mixed_damage,data/processed/clean/p002_clean.png,data/processed/masked/p002_mixed_damage_damage...,data/processed/restored/stable_diffusion/canon...,data/processed/masks/p002_mixed_damage_mask.png,159.0,0.0,608.0,768.0
8,sd__can__p002__scratch_thin__p00_generic__s202...,sd__can__p002__scratch_thin__p00_generic__s202...,p002_scratch_thin,scratch_thin,data/processed/clean/p002_clean.png,data/processed/masked/p002_scratch_thin_damage...,data/processed/restored/stable_diffusion/canon...,data/processed/masks/p002_scratch_thin_mask.png,159.0,0.0,608.0,768.0
9,sd__can__p002__zero__p00_generic__s2026__606dc...,sd__can__p002__zero__p00_generic__s2026__606dc...,p002_zero_control,zero_control,data/processed/clean/p002_clean.png,data/processed/masked/p002_zero_control_damage...,data/processed/restored/stable_diffusion/canon...,data/processed/masks/p002_zero_control_mask.png,159.0,0.0,608.0,768.0


In [8]:
# Batch 1 / Cell 8 - Normalize paths, verify images/masks, and preserve content boxes
def normalise_path_value(path_value) -> str:
    if pd.isna(path_value):
        return ""
    raw_value = str(path_value).strip()
    if raw_value == "":
        return ""
    path = Path(raw_value)
    if path.is_absolute():
        try:
            return path.resolve().relative_to(PROJECT_ROOT).as_posix()
        except ValueError:
            return path.as_posix()
    return path.as_posix()


def path_exists_and_nonempty(path_value) -> bool:
    if pd.isna(path_value) or str(path_value).strip() == "":
        return False
    path = resolve_project_path(str(path_value))
    return path.is_file() and path.stat().st_size > 0


def read_image_size(path_value) -> tuple[int | None, int | None]:
    if pd.isna(path_value) or str(path_value).strip() == "":
        return None, None
    path = resolve_project_path(str(path_value))
    if not path.is_file():
        return None, None
    with Image.open(path) as image:
        return int(image.width), int(image.height)


def read_mask_summary(path_value, threshold: int = MASK_BINARY_THRESHOLD) -> dict[str, Any]:
    if pd.isna(path_value) or str(path_value).strip() == "":
        return {
            "mask_file_exists": False,
            "mask_width": None,
            "mask_height": None,
            "mask_area_pixels_observed": None,
            "has_masked_region": False,
        }

    path = resolve_project_path(str(path_value))
    if not path.is_file():
        return {
            "mask_file_exists": False,
            "mask_width": None,
            "mask_height": None,
            "mask_area_pixels_observed": None,
            "has_masked_region": False,
        }

    mask_bool = metrics_lpips.load_mask_bool(path, threshold=threshold)
    return {
        "mask_file_exists": True,
        "mask_width": int(mask_bool.shape[1]),
        "mask_height": int(mask_bool.shape[0]),
        "mask_area_pixels_observed": int(mask_bool.sum()),
        "has_masked_region": bool(mask_bool.any()),
    }


CONTENT_COORD_COLUMNS = [
    "content_x_min",
    "content_y_min",
    "content_x_max",
    "content_y_max",
]

CONTENT_BOX_SOURCE_MAPS = [
    (
        "notebook22_content_box",
        {
            "content_x_min": "content_x_min",
            "content_y_min": "content_y_min",
            "content_x_max": "content_x_max",
            "content_y_max": "content_y_max",
        },
    ),
    (
        "clean_metadata_content_box",
        {
            "content_x_min": "content_x_min_clean_metadata",
            "content_y_min": "content_y_min_clean_metadata",
            "content_x_max": "content_x_max_clean_metadata",
            "content_y_max": "content_y_max_clean_metadata",
        },
    ),
    (
        "content_bbox_columns",
        {
            "content_x_min": "content_bbox_left",
            "content_y_min": "content_bbox_top",
            "content_x_max": "content_bbox_right",
            "content_y_max": "content_bbox_bottom",
        },
    ),
]


def resolve_lpips_content_box(row: pd.Series) -> dict[str, Any]:
    width = row.get("clean_width_observed")
    height = row.get("clean_height_observed")

    if pd.isna(width) or pd.isna(height):
        return {
            "content_x_min": pd.NA,
            "content_y_min": pd.NA,
            "content_x_max": pd.NA,
            "content_y_max": pd.NA,
            "content_box_source": "unresolved",
            "content_box_fallback_full_image": False,
            "content_box_issue": "missing image size",
        }

    width = int(width)
    height = int(height)
    last_issue = "missing content box in all known sources"

    for source_name, column_map in CONTENT_BOX_SOURCE_MAPS:
        if not all(source_column in row.index for source_column in column_map.values()):
            continue

        values = {}
        complete = True

        for target_column, source_column in column_map.items():
            value = pd.to_numeric(row[source_column], errors="coerce")
            if pd.isna(value):
                complete = False
                break
            values[target_column] = int(value)

        if not complete:
            continue

        trial_row = row.copy()
        for target_column, value in values.items():
            trial_row[target_column] = value

        try:
            x_min, y_min, x_max, y_max = metrics_lpips.get_content_box_from_row(
                trial_row,
                image_size=(width, height),
            )
            return {
                "content_x_min": int(x_min),
                "content_y_min": int(y_min),
                "content_x_max": int(x_max),
                "content_y_max": int(y_max),
                "content_box_source": source_name,
                "content_box_fallback_full_image": False,
                "content_box_issue": "",
            }
        except Exception as exc:
            last_issue = f"invalid {source_name}: {exc}"

    return {
        "content_x_min": 0,
        "content_y_min": 0,
        "content_x_max": width,
        "content_y_max": height,
        "content_box_source": "fallback_full_image",
        "content_box_fallback_full_image": True,
        "content_box_issue": last_issue,
    }


def content_box_usable_after_helper_clip(row: pd.Series) -> bool:
    try:
        width = int(row["clean_width_observed"])
        height = int(row["clean_height_observed"])
        metrics_lpips.get_content_box_from_row(row, image_size=(width, height))
        return True
    except Exception:
        return False


input_cases_df = candidate_manifest_df.copy()

for column in CONTENT_COORD_COLUMNS:
    input_cases_df[f"source_{column}"] = (
        input_cases_df[column] if column in input_cases_df.columns else pd.NA
    )

for path_column in ["clean_path", "damaged_path", "restored_path", "mask_path"]:
    input_cases_df[path_column] = input_cases_df[path_column].map(normalise_path_value)
    input_cases_df[f"{path_column}_exists"] = input_cases_df[path_column].map(path_exists_and_nonempty)

clean_sizes = input_cases_df["clean_path"].map(read_image_size)
damaged_sizes = input_cases_df["damaged_path"].map(read_image_size)
restored_sizes = input_cases_df["restored_path"].map(read_image_size)

input_cases_df[["clean_width_observed", "clean_height_observed"]] = pd.DataFrame(clean_sizes.tolist(), index=input_cases_df.index)
input_cases_df[["damaged_width_observed", "damaged_height_observed"]] = pd.DataFrame(damaged_sizes.tolist(), index=input_cases_df.index)
input_cases_df[["restored_width_observed", "restored_height_observed"]] = pd.DataFrame(restored_sizes.tolist(), index=input_cases_df.index)

mask_summary_df = pd.DataFrame(
    input_cases_df["mask_path"].map(read_mask_summary).tolist(),
    index=input_cases_df.index,
)
input_cases_df = pd.concat([input_cases_df, mask_summary_df], axis=1)

content_box_df = pd.DataFrame(
    input_cases_df.apply(resolve_lpips_content_box, axis=1).tolist(),
    index=input_cases_df.index,
)

for column in CONTENT_COORD_COLUMNS:
    input_cases_df[column] = content_box_df[column].astype("Int64")

input_cases_df["content_box_source"] = content_box_df["content_box_source"]
input_cases_df["content_box_fallback_full_image"] = content_box_df["content_box_fallback_full_image"].astype(bool)
input_cases_df["content_box_issue"] = content_box_df["content_box_issue"]

input_cases_df["image_sizes_match"] = (
    input_cases_df["clean_width_observed"].eq(input_cases_df["damaged_width_observed"])
    & input_cases_df["clean_height_observed"].eq(input_cases_df["damaged_height_observed"])
    & input_cases_df["clean_width_observed"].eq(input_cases_df["restored_width_observed"])
    & input_cases_df["clean_height_observed"].eq(input_cases_df["restored_height_observed"])
)

input_cases_df["target_size_valid"] = (
    input_cases_df["clean_width_observed"].eq(TARGET_SIZE)
    & input_cases_df["clean_height_observed"].eq(TARGET_SIZE)
)

input_cases_df["mask_size_matches_image"] = (
    input_cases_df["mask_width"].eq(input_cases_df["clean_width_observed"])
    & input_cases_df["mask_height"].eq(input_cases_df["clean_height_observed"])
)

input_cases_df["content_box_usable"] = input_cases_df.apply(content_box_usable_after_helper_clip, axis=1)

input_cases_df["is_zero_control"] = input_cases_df["mask_type"].astype(str).str.strip().eq("zero_control")
input_cases_df["mask_bbox_applicable"] = input_cases_df["has_masked_region"].astype(bool)

input_cases_df["empty_nonzero_mask"] = (
    ~input_cases_df["is_zero_control"].astype(bool)
    & ~input_cases_df["has_masked_region"].astype(bool)
)

input_cases_df["expected_lpips_region_count"] = np.where(
    input_cases_df["mask_bbox_applicable"].astype(bool),
    3,
    2,
).astype(int)

input_cases_df["expected_lpips_regions"] = np.where(
    input_cases_df["mask_bbox_applicable"].astype(bool),
    "full_image|content_region|mask_bbox_crop",
    "full_image|content_region",
)

if "metric_case_id" in input_cases_df.columns:
    input_cases_df["upstream_metric_case_id"] = input_cases_df["metric_case_id"].astype(str)

input_cases_df["metric_case_id"] = [
    f"sd24_{index:04d}" for index in range(1, len(input_cases_df) + 1)
]
input_cases_df["lpips_case_id"] = input_cases_df["metric_case_id"]

if "source_case_id" not in input_cases_df.columns:
    input_cases_df["source_case_id"] = input_cases_df["case_id"].astype(str)

if "source_case_id_original" not in input_cases_df.columns:
    input_cases_df["source_case_id_original"] = input_cases_df["source_case_id"].astype(str)

if "metric_applicability" not in input_cases_df.columns:
    input_cases_df["metric_applicability"] = "primary"

input_cases_df["batch1_ready_for_lpips"] = (
    input_cases_df["clean_path_exists"].astype(bool)
    & input_cases_df["damaged_path_exists"].astype(bool)
    & input_cases_df["restored_path_exists"].astype(bool)
    & input_cases_df["mask_path_exists"].astype(bool)
    & input_cases_df["mask_file_exists"].astype(bool)
    & input_cases_df["image_sizes_match"].astype(bool)
    & input_cases_df["target_size_valid"].astype(bool)
    & input_cases_df["mask_size_matches_image"].astype(bool)
    & input_cases_df["content_box_usable"].astype(bool)
    & ~input_cases_df["content_box_fallback_full_image"].astype(bool)
)

OBSERVED_MASK_BBOX_ROWS = int(input_cases_df["mask_bbox_applicable"].astype(bool).sum())
OBSERVED_NO_MASK_BBOX_ROWS = int((~input_cases_df["mask_bbox_applicable"].astype(bool)).sum())
OBSERVED_LPIPS_ROWS = int(input_cases_df["expected_lpips_region_count"].sum())

EXPECTED_REGION_COUNTS = {
    "full_image": EXPECTED_CANDIDATE_ROWS,
    "content_region": EXPECTED_CANDIDATE_ROWS,
    "mask_bbox_crop": OBSERVED_MASK_BBOX_ROWS,
}

print(f"Candidate rows: {len(input_cases_df):,}")
print(f"Mask-bbox applicable rows: {OBSERVED_MASK_BBOX_ROWS:,}")
print(f"No mask-bbox rows: {OBSERVED_NO_MASK_BBOX_ROWS:,}")
print(f"Expected LPIPS rows: {OBSERVED_LPIPS_ROWS:,}")
print(f"Empty non-zero-control masks: {int(input_cases_df['empty_nonzero_mask'].sum()):,}")

display(input_cases_df["content_box_source"].value_counts(dropna=False).rename_axis("content_box_source").reset_index(name="rows"))

Candidate rows: 945
Mask-bbox applicable rows: 834
No mask-bbox rows: 111
Expected LPIPS rows: 2,724
Empty non-zero-control masks: 17


,content_box_source,rows
0,notebook22_content_box,945


In [9]:
# Batch 1 / Cell 9 - Summaries, diagnostics, and save LPIPS input cases
batch1_summary = {
    "input_case_rows": int(len(input_cases_df)),
    "unique_candidate_ids": int(input_cases_df["candidate_id"].nunique()),
    "unique_metric_case_ids": int(input_cases_df["metric_case_id"].nunique()),
    "zero_control_rows": int(input_cases_df["is_zero_control"].astype(bool).sum()),
    "mask_bbox_applicable_rows": OBSERVED_MASK_BBOX_ROWS,
    "no_mask_bbox_rows": OBSERVED_NO_MASK_BBOX_ROWS,
    "empty_nonzero_mask_rows": int(input_cases_df["empty_nonzero_mask"].astype(bool).sum()),
    "expected_lpips_rows": OBSERVED_LPIPS_ROWS,
    "ready_for_lpips_rows": int(input_cases_df["batch1_ready_for_lpips"].astype(bool).sum()),
    "not_ready_for_lpips_rows": int((~input_cases_df["batch1_ready_for_lpips"].astype(bool)).sum()),
    "content_box_fallback_full_image_rows": int(input_cases_df["content_box_fallback_full_image"].astype(bool).sum()),
}

batch1_summary_df = pd.DataFrame([{"metric": key, "value": value} for key, value in batch1_summary.items()])

path_existence_summary_df = pd.DataFrame(
    [
        {
            "path_type": path_column,
            "existing_nonempty_files": int(input_cases_df[f"{path_column}_exists"].astype(bool).sum()),
            "missing_or_empty_files": int((~input_cases_df[f"{path_column}_exists"].astype(bool)).sum()),
        }
        for path_column in ["clean_path", "damaged_path", "restored_path", "mask_path"]
    ]
)

region_count_summary_df = (
    input_cases_df["expected_lpips_regions"]
    .value_counts()
    .rename_axis("expected_lpips_regions")
    .reset_index(name="case_count")
)

mask_type_summary_df = (
    input_cases_df.groupby(["mask_type", "has_masked_region", "empty_nonzero_mask"], dropna=False)
    .size()
    .reset_index(name="case_count")
    .sort_values(["mask_type", "has_masked_region", "empty_nonzero_mask"], kind="stable")
)

content_box_summary_df = (
    input_cases_df.groupby(["content_box_source", "content_box_fallback_full_image"], dropna=False)
    .size()
    .reset_index(name="case_count")
    .sort_values(["content_box_source", "content_box_fallback_full_image"], kind="stable")
)

print("Batch 1 summary:")
display(batch1_summary_df)

print("Path existence summary:")
display(path_existence_summary_df)

print("Expected LPIPS region-pattern summary:")
display(region_count_summary_df)

print("Mask diagnostics:")
display(mask_type_summary_df)

print("Content-box diagnostics:")
display(content_box_summary_df)

if input_cases_df["empty_nonzero_mask"].astype(bool).any():
    print("Non-zero-control rows with empty observed masks. These get full_image + content_region only.")
    display(
        input_cases_df.loc[
            input_cases_df["empty_nonzero_mask"].astype(bool),
            ["metric_case_id", "candidate_id", "case_id", "mask_type", "mask_path", "mask_area_pixels_observed"],
        ].head(25)
    )

if input_cases_df["content_box_fallback_full_image"].astype(bool).any():
    print("Rows missing a real content box. These block Batch 1.")
    display(
        input_cases_df.loc[
            input_cases_df["content_box_fallback_full_image"].astype(bool),
            ["metric_case_id", "candidate_id", "case_id", "content_box_source", "content_box_issue"],
        ].head(25)
    )

preferred_front_columns = [
    "metric_case_id",
    "lpips_case_id",
    "upstream_metric_case_id",
    "candidate_id",
    "candidate_index",
    "candidate_seed",
    "effective_candidate_seed",
    "case_id",
    "source_case_id",
    "source_case_id_original",
    "restoration_case_id",
    "painting_id",
    "category",
    "title",
    "artist",
    "style",
    "model_name",
    "mask_id",
    "mask_type",
    "is_zero_control",
    "has_masked_region",
    "empty_nonzero_mask",
    "mask_bbox_applicable",
    "metric_applicability",
    "prompt_policy_id",
    "prompt_variant_id",
    "prompt_template_name",
    "prompt_variant_family",
    "clean_path",
    "damaged_path",
    "restored_path",
    "mask_path",
    "source_content_x_min",
    "source_content_y_min",
    "source_content_x_max",
    "source_content_y_max",
    "content_x_min",
    "content_y_min",
    "content_x_max",
    "content_y_max",
    "content_box_source",
    "content_box_fallback_full_image",
    "content_box_issue",
    "content_box_usable",
    "mask_area_pixels_observed",
    "expected_lpips_region_count",
    "expected_lpips_regions",
    "batch1_ready_for_lpips",
]

front_columns = [column for column in preferred_front_columns if column in input_cases_df.columns]
remaining_columns = [column for column in input_cases_df.columns if column not in front_columns]

input_cases_df = input_cases_df[front_columns + remaining_columns].copy()
input_cases_df.to_csv(BATCH1_INPUT_CASES_PATH, index=False)

print(f"Saved: {rel(BATCH1_INPUT_CASES_PATH)}")
print(f"Rows: {len(input_cases_df):,}")
print(f"Columns: {len(input_cases_df.columns):,}")

display(
    input_cases_df[
        [
            "metric_case_id",
            "candidate_id",
            "case_id",
            "mask_type",
            "has_masked_region",
            "empty_nonzero_mask",
            "content_box_source",
            "expected_lpips_region_count",
            "batch1_ready_for_lpips",
        ]
    ].head(10)
)

Batch 1 summary:


,metric,value
0,input_case_rows,945
1,unique_candidate_ids,945
2,unique_metric_case_ids,945
3,zero_control_rows,94
4,mask_bbox_applicable_rows,834
5,no_mask_bbox_rows,111
6,empty_nonzero_mask_rows,17
7,expected_lpips_rows,2724
8,ready_for_lpips_rows,945
9,not_ready_for_lpips_rows,0


Path existence summary:


,path_type,existing_nonempty_files,missing_or_empty_files
0,clean_path,945,0
1,damaged_path,945,0
2,restored_path,945,0
3,mask_path,945,0


Expected LPIPS region-pattern summary:


...

,content_box_source,content_box_fallback_full_image,case_count
0,notebook22_content_box,False,945


Non-zero-control rows with empty observed masks. These get full_image + content_region only.


,metric_case_id,candidate_id,case_id,mask_type,mask_path,mask_area_pixels_observed
825,sd24_0826,sd__syn__p026__water_stain__p00_generic__s2026...,p026__water_stain__moderate,water_stain,data/processed/masks/synthetic_degradation/p02...,0
826,sd24_0827,sd__syn__p026__water_stain__p00_generic__s2026...,p026__water_stain__severe,water_stain,data/processed/masks/synthetic_degradation/p02...,0
827,sd24_0828,sd__syn__p026__water_stain__p01_style_period__...,p026__water_stain__severe,water_stain,data/processed/masks/synthetic_degradation/p02...,0
828,sd24_0829,sd__syn__p026__water_stain__p02_artist__s2026_...,p026__water_stain__severe,water_stain,data/processed/masks/synthetic_degradation/p02...,0
829,sd24_0830,sd__syn__p026__water_stain__p03_artist_style_p...,p026__water_stain__severe,water_stain,data/processed/masks/synthetic_degradation/p02...,0
830,sd24_0831,sd__syn__p026__water_stain__p04_full_context__...,p026__water_stain__severe,water_stain,data/processed/masks/synthetic_degradation/p02...,0
876,sd24_0877,sd__syn__p039__water_stain_dirt__p00_generic__...,p039__water_stain_dirt__moderate,water_stain_dirt,data/processed/masks/synthetic_degradation/p03...,0
877,sd24_0878,sd__syn__p039__water_stain_dirt__p01_style_per...,p039__water_stain_dirt__moderate,water_stain_dirt,data/processed/masks/synthetic_degradation/p03...,0
878,sd24_0879,sd__syn__p039__water_stain_dirt__p02_artist__s...,p039__water_stain_dirt__moderate,water_stain_dirt,data/processed/masks/synthetic_degradation/p03...,0
879,sd24_0880,sd__syn__p039__water_stain_dirt__p03_artist_st...,p039__water_stain_dirt__moderate,water_stain_dirt,data/processed/masks/synthetic_degradation/p03...,0


Saved: outputs/24_stable_diffusion_lpips_metrics/tables/stable_diffusion_lpips_input_cases.csv
Rows: 945
Columns: 309


,metric_case_id,candidate_id,case_id,mask_type,has_masked_region,empty_nonzero_mask,content_box_source,expected_lpips_region_count,batch1_ready_for_lpips
0,sd24_0001,sd__can__p001__loss_large__p00_generic__s2026_...,p001_loss_large,loss_large,True,False,notebook22_content_box,3,True
1,sd24_0002,sd__can__p001__loss_small__p00_generic__s2026_...,p001_loss_small,loss_small,True,False,notebook22_content_box,3,True
2,sd24_0003,sd__can__p001__mixed_damage__p00_generic__s202...,p001_mixed_damage,mixed_damage,True,False,notebook22_content_box,3,True
3,sd24_0004,sd__can__p001__scratch_thin__p00_generic__s202...,p001_scratch_thin,scratch_thin,True,False,notebook22_content_box,3,True
4,sd24_0005,sd__can__p001__zero__p00_generic__s2026__3be10...,p001_zero_control,zero_control,False,False,notebook22_content_box,2,True
5,sd24_0006,sd__can__p002__loss_large__p00_generic__s2026_...,p002_loss_large,loss_large,True,False,notebook22_content_box,3,True
6,sd24_0007,sd__can__p002__loss_small__p00_generic__s2026_...,p002_loss_small,loss_small,True,False,notebook22_content_box,3,True
7,sd24_0008,sd__can__p002__mixed_damage__p00_generic__s202...,p002_mixed_damage,mixed_damage,True,False,notebook22_content_box,3,True
8,sd24_0009,sd__can__p002__scratch_thin__p00_generic__s202...,p002_scratch_thin,scratch_thin,True,False,notebook22_content_box,3,True
9,sd24_0010,sd__can__p002__zero__p00_generic__s2026__606dc...,p002_zero_control,zero_control,False,False,notebook22_content_box,2,True


In [10]:
# Batch 1 / Cell 10 - Validate Batch 1 and update stage manifest
saved_input_cases_df = pd.read_csv(BATCH1_INPUT_CASES_PATH)

required_batch1_columns = [
    "metric_case_id",
    "lpips_case_id",
    "candidate_id",
    "case_id",
    "painting_id",
    "model_name",
    "mask_id",
    "mask_type",
    "clean_path",
    "damaged_path",
    "restored_path",
    "mask_path",
    "content_x_min",
    "content_y_min",
    "content_x_max",
    "content_y_max",
    "content_box_source",
    "content_box_fallback_full_image",
    "content_box_usable",
    "has_masked_region",
    "is_zero_control",
    "empty_nonzero_mask",
    "mask_bbox_applicable",
    "expected_lpips_region_count",
    "expected_lpips_regions",
    "batch1_ready_for_lpips",
]

missing_batch1_columns = [
    column for column in required_batch1_columns
    if column not in saved_input_cases_df.columns
]

candidate_id_duplicate_rows = int(saved_input_cases_df.duplicated(["candidate_id"], keep=False).sum())
metric_case_id_duplicate_rows = int(saved_input_cases_df.duplicated(["metric_case_id"], keep=False).sum())

ready_rows = int(bool_series(saved_input_cases_df["batch1_ready_for_lpips"]).sum())
not_ready_rows = int((~bool_series(saved_input_cases_df["batch1_ready_for_lpips"])).sum())

mask_bbox_rows = int(bool_series(saved_input_cases_df["mask_bbox_applicable"]).sum())
no_mask_bbox_rows = int((~bool_series(saved_input_cases_df["mask_bbox_applicable"])).sum())
zero_control_rows = int(bool_series(saved_input_cases_df["is_zero_control"]).sum())
empty_nonzero_mask_rows = int(bool_series(saved_input_cases_df["empty_nonzero_mask"]).sum())
content_box_fallback_rows = int(bool_series(saved_input_cases_df["content_box_fallback_full_image"]).sum())

expected_lpips_rows_from_saved_cases = int(saved_input_cases_df["expected_lpips_region_count"].sum())

path_missing_counts = {
    path_column: int((~bool_series(saved_input_cases_df[f"{path_column}_exists"])).sum())
    for path_column in ["clean_path", "damaged_path", "restored_path", "mask_path"]
    if f"{path_column}_exists" in saved_input_cases_df.columns
}

invalid_image_size_rows = int((~bool_series(saved_input_cases_df["image_sizes_match"])).sum())
invalid_target_size_rows = int((~bool_series(saved_input_cases_df["target_size_valid"])).sum())
invalid_mask_size_rows = int((~bool_series(saved_input_cases_df["mask_size_matches_image"])).sum())
invalid_content_box_rows = int((~bool_series(saved_input_cases_df["content_box_usable"])).sum())

batch1_validation_df = pd.DataFrame(
    [
        validation_row(
            "batch0_validation_passed",
            rel(BATCH0_VALIDATION_PATH),
            "all checks passed",
            BATCH0_VALIDATION_PATH.is_file()
            and bool_series(pd.read_csv(BATCH0_VALIDATION_PATH)["passed"]).all(),
            "Batch 0 validation is missing or did not pass.",
        ),
        validation_row(
            "input_cases_written",
            rel(BATCH1_INPUT_CASES_PATH),
            "file exists",
            BATCH1_INPUT_CASES_PATH.is_file(),
            "LPIPS input cases CSV was not written.",
        ),
        validation_row(
            "input_case_row_count",
            len(saved_input_cases_df),
            EXPECTED_CANDIDATE_ROWS,
            len(saved_input_cases_df) == EXPECTED_CANDIDATE_ROWS,
            "LPIPS input case row count does not match the Notebook 22 contract.",
        ),
        validation_row(
            "required_batch1_columns_present",
            missing_batch1_columns,
            [],
            len(missing_batch1_columns) == 0,
            "LPIPS input cases CSV is missing required columns.",
        ),
        validation_row(
            "candidate_ids_unique",
            candidate_id_duplicate_rows,
            0,
            candidate_id_duplicate_rows == 0,
            "Candidate IDs are not unique.",
        ),
        validation_row(
            "metric_case_ids_unique",
            metric_case_id_duplicate_rows,
            0,
            metric_case_id_duplicate_rows == 0,
            "Notebook 24 metric case IDs are not unique.",
        ),
        validation_row(
            "all_cases_ready_for_lpips",
            ready_rows,
            EXPECTED_CANDIDATE_ROWS,
            ready_rows == EXPECTED_CANDIDATE_ROWS and not_ready_rows == 0,
            "One or more input cases is not ready for LPIPS computation.",
        ),
        validation_row(
            "all_required_files_exist",
            path_missing_counts,
            {"clean_path": 0, "damaged_path": 0, "restored_path": 0, "mask_path": 0},
            all(value == 0 for value in path_missing_counts.values()),
            "One or more clean/damaged/restored/mask files is missing or empty.",
        ),
        validation_row(
            "image_sizes_match",
            invalid_image_size_rows,
            0,
            invalid_image_size_rows == 0,
            "One or more clean/damaged/restored triplets has mismatched dimensions.",
        ),
        validation_row(
            "target_size_valid",
            invalid_target_size_rows,
            0,
            invalid_target_size_rows == 0,
            f"One or more input images is not {TARGET_SIZE}x{TARGET_SIZE}.",
        ),
        validation_row(
            "mask_sizes_match_image",
            invalid_mask_size_rows,
            0,
            invalid_mask_size_rows == 0,
            "One or more masks does not match its image dimensions.",
        ),
        validation_row(
            "content_boxes_usable_by_lpips_helper",
            invalid_content_box_rows,
            0,
            invalid_content_box_rows == 0,
            "One or more content boxes cannot be clipped into a valid LPIPS crop.",
        ),
        validation_row(
            "no_full_image_content_box_fallbacks",
            content_box_fallback_rows,
            0,
            content_box_fallback_rows == 0,
            "One or more rows lacked a real content box. Fix upstream propagation before LPIPS.",
        ),
        validation_row(
            "zero_control_row_count",
            zero_control_rows,
            EXPECTED_ZERO_CONTROL_ROWS,
            zero_control_rows == EXPECTED_ZERO_CONTROL_ROWS,
            "Zero-control row count does not match the Notebook 22/23 contract.",
        ),
        validation_row(
            "mask_bbox_region_count_observed",
            mask_bbox_rows,
            OBSERVED_MASK_BBOX_ROWS,
            mask_bbox_rows == OBSERVED_MASK_BBOX_ROWS,
            "Mask-bbox row count changed after saving input cases.",
        ),
        validation_row(
            "no_mask_bbox_region_count_observed",
            no_mask_bbox_rows,
            OBSERVED_NO_MASK_BBOX_ROWS,
            no_mask_bbox_rows == OBSERVED_NO_MASK_BBOX_ROWS,
            "No-mask-bbox row count changed after saving input cases.",
        ),
        validation_row(
            "empty_nonzero_masks_reported",
            empty_nonzero_mask_rows,
            "reported only",
            True,
            "Empty non-zero-control masks are reported and receive no mask_bbox_crop row.",
        ),
        validation_row(
            "expected_lpips_row_count_observed",
            expected_lpips_rows_from_saved_cases,
            OBSERVED_LPIPS_ROWS,
            expected_lpips_rows_from_saved_cases == OBSERVED_LPIPS_ROWS,
            "Expected LPIPS row count changed after saving input cases.",
        ),
    ]
)

batch1_validation_df.to_csv(BATCH1_VALIDATION_PATH, index=False)

batch1_passed = bool_series(batch1_validation_df["passed"]).all()

stage_manifest = json.loads(STAGE_MANIFEST_PATH.read_text(encoding="utf-8"))

stage_manifest["stage"] = "batch1_lpips_input_cases"
stage_manifest["stage_status"] = "passed" if batch1_passed else "failed"
stage_manifest["updated_at_utc"] = utc_now_iso()
stage_manifest["lpips_contract"].update(
    {
        "mask_bbox_applicable_rows": mask_bbox_rows,
        "no_mask_bbox_rows": no_mask_bbox_rows,
        "empty_nonzero_mask_rows": empty_nonzero_mask_rows,
        "expected_region_counts": EXPECTED_REGION_COUNTS,
        "expected_lpips_rows": OBSERVED_LPIPS_ROWS,
    }
)
stage_manifest["batch1"] = {
    "status": "passed" if batch1_passed else "failed",
    "input_cases": rel(BATCH1_INPUT_CASES_PATH),
    "validation": rel(BATCH1_VALIDATION_PATH),
    "summary": batch1_summary,
    "path_missing_counts": path_missing_counts,
    "checks_passed": int(bool_series(batch1_validation_df["passed"]).sum()),
    "checks_total": int(len(batch1_validation_df)),
}

STAGE_MANIFEST_PATH.write_text(json.dumps(to_json_safe(stage_manifest), indent=2), encoding="utf-8")

print(f"Saved: {rel(BATCH1_VALIDATION_PATH)}")
print(f"Updated: {rel(STAGE_MANIFEST_PATH)}")
print(f"Batch 1 checks passed: {int(bool_series(batch1_validation_df['passed']).sum())} / {len(batch1_validation_df)}")

display(batch1_validation_df)

if not batch1_passed:
    display(batch1_validation_df.loc[~bool_series(batch1_validation_df["passed"]), ["check_name", "failure_message"]])
    raise RuntimeError("Batch 1 validation failed.")

Saved: outputs/24_stable_diffusion_lpips_metrics/validation/batch1_input_validation.csv
Updated: outputs/24_stable_diffusion_lpips_metrics/manifests/stable_diffusion_lpips_metrics_stage_manifest.json
Batch 1 checks passed: 18 / 18


,check_name,observed,expected,passed,failure_message
0,batch0_validation_passed,outputs/24_stable_diffusion_lpips_metrics/vali...,all checks passed,True,
1,input_cases_written,outputs/24_stable_diffusion_lpips_metrics/tabl...,file exists,True,
2,input_case_row_count,945,945,True,
3,required_batch1_columns_present,[],[],True,
4,candidate_ids_unique,0,0,True,
5,metric_case_ids_unique,0,0,True,
6,all_cases_ready_for_lpips,945,945,True,
7,all_required_files_exist,"{'clean_path': 0, 'damaged_path': 0, 'restored...","{'clean_path': 0, 'damaged_path': 0, 'restored...",True,
8,image_sizes_match,0,0,True,
9,target_size_valid,0,0,True,


In [11]:
# Batch 2 / Cell 11 - Select deterministic LPIPS smoke cases
import time

if not BATCH1_VALIDATION_PATH.is_file():
    raise FileNotFoundError(f"Batch 1 validation not found: {BATCH1_VALIDATION_PATH}")

batch1_validation_df = pd.read_csv(BATCH1_VALIDATION_PATH)
if not bool_series(batch1_validation_df["passed"]).all():
    display(batch1_validation_df.loc[~bool_series(batch1_validation_df["passed"])])
    raise RuntimeError("Batch 1 did not pass. Do not continue to Batch 2.")

if not BATCH1_INPUT_CASES_PATH.is_file():
    raise FileNotFoundError(f"Batch 1 input cases not found: {BATCH1_INPUT_CASES_PATH}")

lpips_input_cases_df = pd.read_csv(BATCH1_INPUT_CASES_PATH)

ready_cases_df = lpips_input_cases_df.loc[
    bool_series(lpips_input_cases_df["batch1_ready_for_lpips"])
].copy()

if ready_cases_df.empty:
    raise RuntimeError("No Batch 1 cases are ready for LPIPS smoke testing.")

for path_column in ["clean_path", "damaged_path", "restored_path", "mask_path"]:
    ready_cases_df[path_column] = ready_cases_df[path_column].astype(str).str.strip()

if "mask_area_pixels" not in ready_cases_df.columns and "mask_area_pixels_observed" in ready_cases_df.columns:
    ready_cases_df["mask_area_pixels"] = ready_cases_df["mask_area_pixels_observed"]

ready_cases_df["is_zero_control_bool"] = bool_series(ready_cases_df["is_zero_control"])
ready_cases_df["mask_bbox_applicable_bool"] = bool_series(ready_cases_df["mask_bbox_applicable"])

smoke_selected_parts = []

zero_control_smoke_df = (
    ready_cases_df.loc[ready_cases_df["is_zero_control_bool"]]
    .sort_values(["dataset_name", "painting_id", "candidate_id"], kind="stable")
    .head(1)
)
smoke_selected_parts.append(zero_control_smoke_df)

mask_bbox_smoke_df = (
    ready_cases_df.loc[
        ready_cases_df["mask_bbox_applicable_bool"]
        & ~ready_cases_df["is_zero_control_bool"]
    ]
    .sort_values(["mask_type", "dataset_name", "painting_id", "candidate_id"], kind="stable")
    .groupby("mask_type", dropna=False, sort=False)
    .head(1)
    .head(2)
)
smoke_selected_parts.append(mask_bbox_smoke_df)

smoke_cases_df = (
    pd.concat(smoke_selected_parts, ignore_index=False)
    .drop_duplicates(subset=["metric_case_id"], keep="first")
    .sort_values(["is_zero_control_bool", "mask_type", "candidate_id"], kind="stable")
    .reset_index(drop=True)
)

if len(smoke_cases_df) < 3:
    fallback_smoke_df = (
        ready_cases_df.loc[~ready_cases_df["metric_case_id"].isin(smoke_cases_df["metric_case_id"])]
        .sort_values(["mask_bbox_applicable_bool", "mask_type", "candidate_id"], ascending=[False, True, True], kind="stable")
        .head(3 - len(smoke_cases_df))
    )
    smoke_cases_df = (
        pd.concat([smoke_cases_df, fallback_smoke_df], ignore_index=True)
        .drop_duplicates(subset=["metric_case_id"], keep="first")
        .reset_index(drop=True)
    )

if not bool_series(smoke_cases_df["is_zero_control"]).any():
    raise RuntimeError("Smoke selection must include one zero-control case.")

if not bool_series(smoke_cases_df["mask_bbox_applicable"]).any():
    raise RuntimeError("Smoke selection must include at least one mask-bbox-applicable case.")

expected_smoke_region_counts = {
    "full_image": int(len(smoke_cases_df)),
    "content_region": int(len(smoke_cases_df)),
    "mask_bbox_crop": int(bool_series(smoke_cases_df["mask_bbox_applicable"]).sum()),
}
expected_smoke_lpips_rows = int(sum(expected_smoke_region_counts.values()))

print(f"Smoke cases: {len(smoke_cases_df):,}")
print(f"Expected smoke LPIPS rows: {expected_smoke_lpips_rows:,}")
print(f"Expected smoke region counts: {expected_smoke_region_counts}")

display(
    smoke_cases_df[
        [
            "metric_case_id",
            "candidate_id",
            "case_id",
            "mask_type",
            "is_zero_control",
            "mask_bbox_applicable",
            "expected_lpips_region_count",
            "clean_path",
            "restored_path",
            "mask_path",
        ]
    ]
)

Smoke cases: 3
Expected smoke LPIPS rows: 8
Expected smoke region counts: {'full_image': 3, 'content_region': 3, 'mask_bbox_crop': 2}


,metric_case_id,candidate_id,case_id,mask_type,is_zero_control,mask_bbox_applicable,expected_lpips_region_count,clean_path,restored_path,mask_path
0,sd24_0664,sd__syn__p001__blur__p00_generic__s2026__a0008...,p001__blur__severe,blur,False,True,3,data/processed/clean/p001_clean.png,data/processed/restored/stable_diffusion/synth...,data/processed/masks/synthetic_degradation/p00...
1,sd24_0657,sd__syn__p001__blur_fading__p00_generic__s2026...,p001__blur_fading__moderate,blur_fading,False,True,3,data/processed/clean/p001_clean.png,data/processed/restored/stable_diffusion/synth...,data/processed/masks/synthetic_degradation/p00...
2,sd24_0005,sd__can__p001__zero__p00_generic__s2026__3be10...,p001_zero_control,zero_control,True,False,2,data/processed/clean/p001_clean.png,data/processed/restored/stable_diffusion/canon...,data/processed/masks/p001_zero_control_mask.png


In [12]:
# Batch 2 / Cell 12 - Load LPIPS model and run smoke metrics
runtime_dependencies_df = metrics_lpips.validate_lpips_runtime_dependencies()
runtime_dependencies_passed = bool_series(runtime_dependencies_df["passed"]).all()

display(runtime_dependencies_df)

if not runtime_dependencies_passed:
    display(runtime_dependencies_df.loc[~bool_series(runtime_dependencies_df["passed"])])
    raise RuntimeError("LPIPS runtime dependencies are missing. Cannot run Batch 2 smoke test.")

model_load_start = time.perf_counter()
lpips_device = metrics_lpips.get_device(prefer_cuda=True)
lpips_model = metrics_lpips.load_lpips_model(
    net=LPIPS_NET,
    device=lpips_device,
)
lpips_model_load_seconds = time.perf_counter() - model_load_start

print(f"LPIPS model loaded: net={LPIPS_NET}, device={lpips_device}")
print(f"Model load seconds: {lpips_model_load_seconds:.2f}")

smoke_run_id = f"sd_lpips_smoke_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}"

smoke_run_start = time.perf_counter()
smoke_metrics_df = metrics_lpips.compute_lpips_metrics_for_restorations(
    restoration_metadata=smoke_cases_df.drop(
        columns=["is_zero_control_bool", "mask_bbox_applicable_bool"],
        errors="ignore",
    ),
    lpips_model=lpips_model,
    device=lpips_device,
    lpips_net=LPIPS_NET,
    target_size=TARGET_SIZE,
    mask_bbox_margin=MASK_BBOX_MARGIN,
    lpips_input_size=LPIPS_INPUT_SIZE,
    mask_binary_threshold=MASK_BINARY_THRESHOLD,
    progress_every=1,
)
lpips_smoke_runtime_seconds = time.perf_counter() - smoke_run_start

smoke_metrics_df.insert(0, "smoke_run_id", smoke_run_id)
smoke_metrics_df.insert(1, "smoke_selected_case_count", int(len(smoke_cases_df)))
smoke_metrics_df.insert(2, "smoke_expected_lpips_rows", int(expected_smoke_lpips_rows))

smoke_metrics_df.to_csv(BATCH2_SMOKE_METRICS_PATH, index=False)

print(f"Saved: {rel(BATCH2_SMOKE_METRICS_PATH)}")
print(f"Smoke runtime seconds: {lpips_smoke_runtime_seconds:.2f}")
print(f"Smoke rows: {len(smoke_metrics_df):,}")

display(smoke_metrics_df["evaluation_region"].value_counts(dropna=False).rename_axis("evaluation_region").reset_index(name="rows"))
display(smoke_metrics_df["status"].value_counts(dropna=False).rename_axis("status").reset_index(name="rows"))

display(
    smoke_metrics_df[
        [
            "lpips_row_id",
            "lpips_case_id",
            "candidate_id",
            "mask_type",
            "is_zero_control",
            "evaluation_region",
            "damaged_lpips",
            "restored_lpips",
            "lpips_improvement",
            "status",
            "issue",
        ]
    ]
)

,component,module,version,required,installed,passed
0,torch,torch,2.5.1+cu121,True,True,True
1,lpips,lpips,0.1.4,True,True,True
2,Pillow,PIL,11.0.0,True,True,True


Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]


C:\Users\rahul\AppData\Local\Programs\Python\Python312\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
C:\Users\rahul\AppData\Local\Programs\Python\Python312\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=AlexNet_Weights.IMAGENET1K_V1`. You can also use `weights=AlexNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Loading model from: C:\Users\rahul\AppData\Local\Programs\Python\Python312\Lib\site-packages\lpips\weights\v0.1\alex.pth


C:\Users\rahul\AppData\Local\Programs\Python\Python312\Lib\site-packages\lpips\lpips.py:107: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.load_state_dict(torch.load(mo

LPIPS model loaded: net=alex, device=cuda
Model load seconds: 23.53
Starting LPIPS metric computation
  Cases: 3
  Target size: (768, 768)
  LPIPS net: alex
  LPIPS input size: 256
  Device: cuda
Computing LPIPS case 1/3 (p001_zero_control) | elapsed 0.01s
Computing LPIPS case 2/3 (p001__blur__severe) | elapsed 1.44s
Computing LPIPS case 3/3 (p001__blur_fading__moderate) | elapsed 1.87s
LPIPS metric computation complete
  Runtime: 2.24 seconds
  Output rows: 8
  Region counts:
evaluation_region
full_image        3
content_region    3
mask_bbox_crop    2
  Status counts:
status
ok    8
Saved: outputs/24_stable_diffusion_lpips_metrics/validation/stable_diffusion_lpips_smoke.csv
Smoke runtime seconds: 2.25
Smoke rows: 8


,evaluation_region,rows
0,full_image,3
1,content_region,3
2,mask_bbox_crop,2


,status,rows
0,ok,8


,lpips_row_id,lpips_case_id,candidate_id,mask_type,is_zero_control,evaluation_region,damaged_lpips,restored_lpips,lpips_improvement,status,issue
0,sd24_0005__full_image__lpips,sd24_0005,sd__can__p001__zero__p00_generic__s2026__3be10...,zero_control,True,full_image,0.000000,0.000000,0.000000,ok,
1,sd24_0005__content_region__lpips,sd24_0005,sd__can__p001__zero__p00_generic__s2026__3be10...,zero_control,True,content_region,0.000000,0.000000,0.000000,ok,
2,sd24_0664__full_image__lpips,sd24_0664,sd__syn__p001__blur__p00_generic__s2026__a0008...,blur,False,full_image,0.092362,0.320604,-0.228242,ok,
3,sd24_0664__content_region__lpips,sd24_0664,sd__syn__p001__blur__p00_generic__s2026__a0008...,blur,False,content_region,0.114639,0.358647,-0.244007,ok,
4,sd24_0664__mask_bbox_crop__lpips,sd24_0664,sd__syn__p001__blur__p00_generic__s2026__a0008...,blur,False,mask_bbox_crop,0.097229,0.328001,-0.230772,ok,
5,sd24_0657__full_image__lpips,sd24_0657,sd__syn__p001__blur_fading__p00_generic__s2026...,blur_fading,False,full_image,0.015497,0.124253,-0.108755,ok,
6,sd24_0657__content_region__lpips,sd24_0657,sd__syn__p001__blur_fading__p00_generic__s2026...,blur_fading,False,content_region,0.020508,0.134697,-0.114190,ok,
7,sd24_0657__mask_bbox_crop__lpips,sd24_0657,sd__syn__p001__blur_fading__p00_generic__s2026...,blur_fading,False,mask_bbox_crop,0.079173,0.328825,-0.249652,ok,


In [13]:
# Batch 2 / Cell 13 - Validate smoke metrics and update stage manifest
def batch2_validation_row(check_name: str, observed, expected, passed: bool, failure_message: str = "") -> dict[str, Any]:
    return {
        "check_name": check_name,
        "observed": observed,
        "expected": expected,
        "passed": bool(passed),
        "failure_message": "" if passed else failure_message,
    }


if not BATCH2_SMOKE_METRICS_PATH.is_file():
    raise FileNotFoundError(f"Smoke metrics CSV not found: {BATCH2_SMOKE_METRICS_PATH}")

saved_smoke_metrics_df = pd.read_csv(BATCH2_SMOKE_METRICS_PATH)
saved_input_cases_df = pd.read_csv(BATCH1_INPUT_CASES_PATH)

smoke_metric_case_ids = sorted(saved_smoke_metrics_df["lpips_case_id"].dropna().astype(str).unique())
selected_input_cases_df = saved_input_cases_df.loc[
    saved_input_cases_df["metric_case_id"].astype(str).isin(smoke_metric_case_ids)
].copy()

if selected_input_cases_df.empty:
    raise RuntimeError("Could not map smoke LPIPS rows back to Batch 1 input cases.")

expected_smoke_region_counts_from_saved = {
    "full_image": int(len(selected_input_cases_df)),
    "content_region": int(len(selected_input_cases_df)),
    "mask_bbox_crop": int(bool_series(selected_input_cases_df["mask_bbox_applicable"]).sum()),
}
expected_smoke_lpips_rows_from_saved = int(sum(expected_smoke_region_counts_from_saved.values()))

helper_smoke_validation_df = metrics_lpips.validate_lpips_metrics(
    saved_smoke_metrics_df,
    expected_rows=expected_smoke_lpips_rows_from_saved,
    expected_region_counts=expected_smoke_region_counts_from_saved,
)

helper_smoke_validation_normalized_df = helper_smoke_validation_df.rename(
    columns={
        "check": "check_name",
        "detail": "failure_message",
    }
).copy()
helper_smoke_validation_normalized_df["observed"] = ""
helper_smoke_validation_normalized_df["expected"] = ""
helper_smoke_validation_normalized_df["validation_source"] = "metrics_lpips.validate_lpips_metrics"

status_error_rows = int(saved_smoke_metrics_df["status"].astype(str).eq("error").sum())
ok_rows_df = saved_smoke_metrics_df.loc[saved_smoke_metrics_df["status"].astype(str).eq("ok")].copy()

numeric_columns = ["damaged_lpips", "restored_lpips", "lpips_improvement"]
finite_numeric_values = bool(
    np.isfinite(
        ok_rows_df[numeric_columns].apply(pd.to_numeric, errors="coerce").to_numpy(dtype=float)
    ).all()
) if not ok_rows_df.empty else False

actual_region_counts = saved_smoke_metrics_df["evaluation_region"].value_counts().to_dict()
actual_region_counts = {str(key): int(value) for key, value in actual_region_counts.items()}

unexpected_region_counts = {
    region: {
        "expected": int(expected_count),
        "actual": int(actual_region_counts.get(region, 0)),
    }
    for region, expected_count in expected_smoke_region_counts_from_saved.items()
    if int(actual_region_counts.get(region, 0)) != int(expected_count)
}

zero_control_case_ids = set(
    selected_input_cases_df.loc[
        bool_series(selected_input_cases_df["is_zero_control"]),
        "metric_case_id",
    ].astype(str)
)
zero_control_mask_bbox_rows = int(
    saved_smoke_metrics_df["lpips_case_id"].astype(str).isin(zero_control_case_ids)
    .where(saved_smoke_metrics_df["evaluation_region"].astype(str).eq("mask_bbox_crop"), False)
    .sum()
)

manual_batch2_validation_df = pd.DataFrame(
    [
        batch2_validation_row(
            "batch1_validation_passed",
            rel(BATCH1_VALIDATION_PATH),
            "all checks passed",
            BATCH1_VALIDATION_PATH.is_file()
            and bool_series(pd.read_csv(BATCH1_VALIDATION_PATH)["passed"]).all(),
            "Batch 1 validation is missing or did not pass.",
        ),
        batch2_validation_row(
            "lpips_model_loaded",
            str(globals().get("lpips_device", "")),
            "model object and device available",
            "lpips_model" in globals() and "lpips_device" in globals(),
            "LPIPS model/device variables are missing. Re-run Batch 2 Cell 12.",
        ),
        batch2_validation_row(
            "smoke_metrics_written",
            rel(BATCH2_SMOKE_METRICS_PATH),
            "file exists",
            BATCH2_SMOKE_METRICS_PATH.is_file(),
            "Smoke metrics CSV was not written.",
        ),
        batch2_validation_row(
            "smoke_case_mapping_to_batch1",
            len(selected_input_cases_df),
            len(smoke_metric_case_ids),
            len(selected_input_cases_df) == len(smoke_metric_case_ids),
            "One or more smoke LPIPS case IDs could not be mapped to Batch 1 input cases.",
        ),
        batch2_validation_row(
            "smoke_row_count",
            len(saved_smoke_metrics_df),
            expected_smoke_lpips_rows_from_saved,
            len(saved_smoke_metrics_df) == expected_smoke_lpips_rows_from_saved,
            "Smoke LPIPS row count does not match selected-case region expectations.",
        ),
        batch2_validation_row(
            "smoke_region_counts",
            actual_region_counts,
            expected_smoke_region_counts_from_saved,
            len(unexpected_region_counts) == 0,
            f"Unexpected smoke region counts: {unexpected_region_counts}",
        ),
        batch2_validation_row(
            "smoke_has_zero_control_case",
            int(bool_series(selected_input_cases_df["is_zero_control"]).sum()),
            ">= 1",
            bool_series(selected_input_cases_df["is_zero_control"]).any(),
            "Smoke sample did not include a zero-control case.",
        ),
        batch2_validation_row(
            "smoke_has_mask_bbox_case",
            int(bool_series(selected_input_cases_df["mask_bbox_applicable"]).sum()),
            ">= 1",
            bool_series(selected_input_cases_df["mask_bbox_applicable"]).any(),
            "Smoke sample did not include a mask-bbox-applicable case.",
        ),
        batch2_validation_row(
            "zero_control_has_no_mask_bbox_crop",
            zero_control_mask_bbox_rows,
            0,
            zero_control_mask_bbox_rows == 0,
            "Zero-control smoke case produced a mask_bbox_crop row.",
        ),
        batch2_validation_row(
            "no_smoke_error_rows",
            status_error_rows,
            0,
            status_error_rows == 0,
            "Smoke LPIPS computation produced one or more error rows.",
        ),
        batch2_validation_row(
            "finite_smoke_lpips_values",
            finite_numeric_values,
            True,
            finite_numeric_values,
            "One or more successful smoke LPIPS values is non-finite.",
        ),
    ]
)

manual_batch2_validation_df["validation_source"] = "notebook_batch2_manual_checks"

batch2_validation_df = pd.concat(
    [
        manual_batch2_validation_df,
        helper_smoke_validation_normalized_df[
            ["check_name", "observed", "expected", "passed", "failure_message", "validation_source"]
        ],
    ],
    ignore_index=True,
)

batch2_validation_df.to_csv(BATCH2_VALIDATION_PATH, index=False)

batch2_passed = bool_series(batch2_validation_df["passed"]).all()

stage_manifest = json.loads(STAGE_MANIFEST_PATH.read_text(encoding="utf-8"))

stage_manifest["stage"] = "batch2_lpips_dependency_model_load_smoke"
stage_manifest["stage_status"] = "passed" if batch2_passed else "failed"
stage_manifest["updated_at_utc"] = utc_now_iso()
stage_manifest["batch2"] = {
    "status": "passed" if batch2_passed else "failed",
    "smoke_metrics": rel(BATCH2_SMOKE_METRICS_PATH),
    "validation": rel(BATCH2_VALIDATION_PATH),
    "smoke_run_id": saved_smoke_metrics_df["smoke_run_id"].iloc[0] if "smoke_run_id" in saved_smoke_metrics_df.columns and len(saved_smoke_metrics_df) else "",
    "smoke_case_count": int(len(selected_input_cases_df)),
    "smoke_case_ids": smoke_metric_case_ids,
    "expected_lpips_rows": int(expected_smoke_lpips_rows_from_saved),
    "expected_region_counts": expected_smoke_region_counts_from_saved,
    "actual_region_counts": actual_region_counts,
    "lpips_net": LPIPS_NET,
    "lpips_input_size": int(LPIPS_INPUT_SIZE),
    "mask_bbox_margin": int(MASK_BBOX_MARGIN),
    "mask_binary_threshold": int(MASK_BINARY_THRESHOLD),
    "target_size": int(TARGET_SIZE),
    "device": str(globals().get("lpips_device", "")),
    "model_load_seconds": float(globals().get("lpips_model_load_seconds", np.nan)),
    "smoke_runtime_seconds": float(globals().get("lpips_smoke_runtime_seconds", np.nan)),
    "checks_passed": int(bool_series(batch2_validation_df["passed"]).sum()),
    "checks_total": int(len(batch2_validation_df)),
}

STAGE_MANIFEST_PATH.write_text(json.dumps(to_json_safe(stage_manifest), indent=2), encoding="utf-8")

print(f"Saved: {rel(BATCH2_VALIDATION_PATH)}")
print(f"Updated: {rel(STAGE_MANIFEST_PATH)}")
print(f"Batch 2 checks passed: {int(bool_series(batch2_validation_df['passed']).sum())} / {len(batch2_validation_df)}")

display(batch2_validation_df)

if not batch2_passed:
    display(batch2_validation_df.loc[~bool_series(batch2_validation_df["passed"]), ["check_name", "failure_message"]])
    raise RuntimeError("Batch 2 validation failed.")

Saved: outputs/24_stable_diffusion_lpips_metrics/validation/batch2_smoke_validation.csv
Updated: outputs/24_stable_diffusion_lpips_metrics/manifests/stable_diffusion_lpips_metrics_stage_manifest.json
Batch 2 checks passed: 19 / 19


,check_name,observed,expected,passed,failure_message,validation_source
0,batch1_validation_passed,outputs/24_stable_diffusion_lpips_metrics/vali...,all checks passed,True,,notebook_batch2_manual_checks
1,lpips_model_loaded,cuda,model object and device available,True,,notebook_batch2_manual_checks
2,smoke_metrics_written,outputs/24_stable_diffusion_lpips_metrics/vali...,file exists,True,,notebook_batch2_manual_checks
3,smoke_case_mapping_to_batch1,3,3,True,,notebook_batch2_manual_checks
4,smoke_row_count,8,8,True,,notebook_batch2_manual_checks
5,smoke_region_counts,"{'full_image': 3, 'content_region': 3, 'mask_b...","{'full_image': 3, 'content_region': 3, 'mask_b...",True,,notebook_batch2_manual_checks
6,smoke_has_zero_control_case,1,>= 1,True,,notebook_batch2_manual_checks
7,smoke_has_mask_bbox_case,2,>= 1,True,,notebook_batch2_manual_checks
8,zero_control_has_no_mask_bbox_crop,0,0,True,,notebook_batch2_manual_checks
9,no_smoke_error_rows,0,0,True,,notebook_batch2_manual_checks


In [14]:
# Batch 3 / Cell 14 - Load full LPIPS input cases and gate on Batch 2 smoke validation
import time

FORCE_RECOMPUTE_FULL_LPIPS = False
FULL_LPIPS_PROGRESS_EVERY = 25

if not BATCH2_VALIDATION_PATH.is_file():
    raise FileNotFoundError(
        f"Batch 2 validation not found yet: {BATCH2_VALIDATION_PATH}. "
        "Wait for Batch 2 to finish and pass before running Batch 3."
    )

batch2_validation_df = pd.read_csv(BATCH2_VALIDATION_PATH)
if not bool_series(batch2_validation_df["passed"]).all():
    display(batch2_validation_df.loc[~bool_series(batch2_validation_df["passed"])])
    raise RuntimeError("Batch 2 did not pass. Do not continue to Batch 3.")

if not BATCH1_INPUT_CASES_PATH.is_file():
    raise FileNotFoundError(f"Batch 1 input cases not found: {BATCH1_INPUT_CASES_PATH}")

full_lpips_input_cases_df = pd.read_csv(BATCH1_INPUT_CASES_PATH)

full_lpips_ready_cases_df = full_lpips_input_cases_df.loc[
    bool_series(full_lpips_input_cases_df["batch1_ready_for_lpips"])
].copy()

if len(full_lpips_ready_cases_df) != EXPECTED_CANDIDATE_ROWS:
    raise RuntimeError(
        f"Expected {EXPECTED_CANDIDATE_ROWS:,} ready cases, "
        f"found {len(full_lpips_ready_cases_df):,}."
    )

for path_column in ["clean_path", "damaged_path", "restored_path", "mask_path"]:
    full_lpips_ready_cases_df[path_column] = (
        full_lpips_ready_cases_df[path_column].astype(str).str.strip()
    )

if "mask_area_pixels" not in full_lpips_ready_cases_df.columns and "mask_area_pixels_observed" in full_lpips_ready_cases_df.columns:
    full_lpips_ready_cases_df["mask_area_pixels"] = full_lpips_ready_cases_df["mask_area_pixels_observed"]

full_expected_region_counts = {
    "full_image": int(len(full_lpips_ready_cases_df)),
    "content_region": int(len(full_lpips_ready_cases_df)),
    "mask_bbox_crop": int(bool_series(full_lpips_ready_cases_df["mask_bbox_applicable"]).sum()),
}
full_expected_lpips_rows = int(sum(full_expected_region_counts.values()))

print(f"Full LPIPS candidate cases: {len(full_lpips_ready_cases_df):,}")
print(f"Expected full LPIPS rows: {full_expected_lpips_rows:,}")
print(f"Expected region counts: {full_expected_region_counts}")

display(
    full_lpips_ready_cases_df[
        [
            "metric_case_id",
            "candidate_id",
            "case_id",
            "mask_type",
            "is_zero_control",
            "mask_bbox_applicable",
            "expected_lpips_region_count",
            "clean_path",
            "restored_path",
            "mask_path",
        ]
    ].head(10)
)

Full LPIPS candidate cases: 945
Expected full LPIPS rows: 2,724
Expected region counts: {'full_image': 945, 'content_region': 945, 'mask_bbox_crop': 834}


,metric_case_id,candidate_id,case_id,mask_type,is_zero_control,mask_bbox_applicable,expected_lpips_region_count,clean_path,restored_path,mask_path
0,sd24_0001,sd__can__p001__loss_large__p00_generic__s2026_...,p001_loss_large,loss_large,False,True,3,data/processed/clean/p001_clean.png,data/processed/restored/stable_diffusion/canon...,data/processed/masks/p001_loss_large_mask.png
1,sd24_0002,sd__can__p001__loss_small__p00_generic__s2026_...,p001_loss_small,loss_small,False,True,3,data/processed/clean/p001_clean.png,data/processed/restored/stable_diffusion/canon...,data/processed/masks/p001_loss_small_mask.png
2,sd24_0003,sd__can__p001__mixed_damage__p00_generic__s202...,p001_mixed_damage,mixed_damage,False,True,3,data/processed/clean/p001_clean.png,data/processed/restored/stable_diffusion/canon...,data/processed/masks/p001_mixed_damage_mask.png
3,sd24_0004,sd__can__p001__scratch_thin__p00_generic__s202...,p001_scratch_thin,scratch_thin,False,True,3,data/processed/clean/p001_clean.png,data/processed/restored/stable_diffusion/canon...,data/processed/masks/p001_scratch_thin_mask.png
4,sd24_0005,sd__can__p001__zero__p00_generic__s2026__3be10...,p001_zero_control,zero_control,True,False,2,data/processed/clean/p001_clean.png,data/processed/restored/stable_diffusion/canon...,data/processed/masks/p001_zero_control_mask.png
5,sd24_0006,sd__can__p002__loss_large__p00_generic__s2026_...,p002_loss_large,loss_large,False,True,3,data/processed/clean/p002_clean.png,data/processed/restored/stable_diffusion/canon...,data/processed/masks/p002_loss_large_mask.png
6,sd24_0007,sd__can__p002__loss_small__p00_generic__s2026_...,p002_loss_small,loss_small,False,True,3,data/processed/clean/p002_clean.png,data/processed/restored/stable_diffusion/canon...,data/processed/masks/p002_loss_small_mask.png
7,sd24_0008,sd__can__p002__mixed_damage__p00_generic__s202...,p002_mixed_damage,mixed_damage,False,True,3,data/processed/clean/p002_clean.png,data/processed/restored/stable_diffusion/canon...,data/processed/masks/p002_mixed_damage_mask.png
8,sd24_0009,sd__can__p002__scratch_thin__p00_generic__s202...,p002_scratch_thin,scratch_thin,False,True,3,data/processed/clean/p002_clean.png,data/processed/restored/stable_diffusion/canon...,data/processed/masks/p002_scratch_thin_mask.png
9,sd24_0010,sd__can__p002__zero__p00_generic__s2026__606dc...,p002_zero_control,zero_control,True,False,2,data/processed/clean/p002_clean.png,data/processed/restored/stable_diffusion/canon...,data/processed/masks/p002_zero_control_mask.png


In [15]:
# Batch 3 / Cell 15 - Run full LPIPS computation
runtime_dependencies_df = metrics_lpips.validate_lpips_runtime_dependencies()
runtime_dependencies_passed = bool_series(runtime_dependencies_df["passed"]).all()

display(runtime_dependencies_df)

if not runtime_dependencies_passed:
    display(runtime_dependencies_df.loc[~bool_series(runtime_dependencies_df["passed"])])
    raise RuntimeError("LPIPS runtime dependencies are missing. Cannot run Batch 3.")

if "lpips_model" in globals() and "lpips_device" in globals():
    print(f"Reusing existing LPIPS model on device: {lpips_device}")
    full_lpips_model_load_seconds = 0.0
else:
    full_model_load_start = time.perf_counter()
    lpips_device = metrics_lpips.get_device(prefer_cuda=True)
    lpips_model = metrics_lpips.load_lpips_model(
        net=LPIPS_NET,
        device=lpips_device,
    )
    full_lpips_model_load_seconds = time.perf_counter() - full_model_load_start
    print(f"Loaded LPIPS model: net={LPIPS_NET}, device={lpips_device}")
    print(f"Model load seconds: {full_lpips_model_load_seconds:.2f}")

full_lpips_run_id = f"sd_lpips_full_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}"

if BATCH3_METRICS_PATH.is_file() and not FORCE_RECOMPUTE_FULL_LPIPS:
    print("Loading existing full LPIPS metrics:")
    print(rel(BATCH3_METRICS_PATH))
    full_lpips_metrics_df = pd.read_csv(BATCH3_METRICS_PATH)
    full_lpips_runtime_seconds = np.nan
else:
    full_lpips_start = time.perf_counter()

    full_lpips_metrics_df = metrics_lpips.compute_lpips_metrics_for_restorations(
        restoration_metadata=full_lpips_ready_cases_df,
        lpips_model=lpips_model,
        device=lpips_device,
        lpips_net=LPIPS_NET,
        target_size=TARGET_SIZE,
        mask_bbox_margin=MASK_BBOX_MARGIN,
        lpips_input_size=LPIPS_INPUT_SIZE,
        mask_binary_threshold=MASK_BINARY_THRESHOLD,
        progress_every=FULL_LPIPS_PROGRESS_EVERY,
    )

    full_lpips_runtime_seconds = time.perf_counter() - full_lpips_start

    full_lpips_metrics_df.insert(0, "full_lpips_run_id", full_lpips_run_id)
    full_lpips_metrics_df.insert(1, "full_input_case_count", int(len(full_lpips_ready_cases_df)))
    full_lpips_metrics_df.insert(2, "full_expected_lpips_rows", int(full_expected_lpips_rows))

    full_lpips_metrics_df.to_csv(BATCH3_METRICS_PATH, index=False)

print(f"Saved/loaded: {rel(BATCH3_METRICS_PATH)}")
print(f"Full LPIPS dataframe shape: {full_lpips_metrics_df.shape}")
print(f"Full LPIPS runtime seconds: {full_lpips_runtime_seconds}")

display(
    full_lpips_metrics_df["evaluation_region"]
    .value_counts(dropna=False)
    .rename_axis("evaluation_region")
    .reset_index(name="rows")
)

display(
    full_lpips_metrics_df["status"]
    .value_counts(dropna=False)
    .rename_axis("status")
    .reset_index(name="rows")
)

display(full_lpips_metrics_df.head())

,component,module,version,required,installed,passed
0,torch,torch,2.5.1+cu121,True,True,True
1,lpips,lpips,0.1.4,True,True,True
2,Pillow,PIL,11.0.0,True,True,True


Reusing existing LPIPS model on device: cuda
Starting LPIPS metric computation
  Cases: 945
  Target size: (768, 768)
  LPIPS net: alex
  LPIPS input size: 256
  Device: cuda
Computing LPIPS case 1/945 (p001_loss_large) | elapsed 0.01s
Computing LPIPS case 25/945 (p004_loss_small) | elapsed 8.59s
Computing LPIPS case 50/945 (p006_zero_control) | elapsed 17.19s
Computing LPIPS case 75/945 (p008_scratch_thin) | elapsed 25.69s
Computing LPIPS case 100/945 (p012_zero_control) | elapsed 34.38s
Computing LPIPS case 125/945 (p014_scratch_thin) | elapsed 42.16s
Computing LPIPS case 150/945 (p018_loss_large) | elapsed 50.18s
Computing LPIPS case 175/945 (p019_zero_control) | elapsed 58.18s
Computing LPIPS case 200/945 (p024_loss_large) | elapsed 67.01s
Computing LPIPS case 225/945 (p025_zero_control) | elapsed 75.24s
Computing LPIPS case 250/945 (p028_mixed_damage) | elapsed 83.53s
Computing LPIPS case 275/945 (p031_zero_control) | elapsed 91.95s
Computing LPIPS case 300/945 (p034_loss_small) |

,evaluation_region,rows
0,full_image,945
1,content_region,945
2,mask_bbox_crop,834


,status,rows
0,ok,2724


,full_lpips_run_id,full_input_case_count,full_expected_lpips_rows,lpips_row_id,lpips_case_id,metric_case_id,candidate_id,restoration_case_id,source_case_id,source_case_id_original,...,prompt_template_name,prompt_variant_family,prompt_variant_order,prompt_ablation_subset,inference_mode,execution_device,scheduler_name,num_inference_steps,guidance_scale,strength
0,sd_lpips_full_20260808T071919Z,945,2724,sd24_0001__full_image__lpips,sd24_0001,sd24_0001,sd__can__p001__loss_large__p00_generic__s2026_...,sd__can__p001__loss_large__p00_generic__s2026_...,p001_loss_large,p001_loss_large,...,generic_restoration,generic,0,False,model_inference,cuda,pipeline_default,30,7.5,1.0
1,sd_lpips_full_20260808T071919Z,945,2724,sd24_0001__content_region__lpips,sd24_0001,sd24_0001,sd__can__p001__loss_large__p00_generic__s2026_...,sd__can__p001__loss_large__p00_generic__s2026_...,p001_loss_large,p001_loss_large,...,generic_restoration,generic,0,False,model_inference,cuda,pipeline_default,30,7.5,1.0
2,sd_lpips_full_20260808T071919Z,945,2724,sd24_0001__mask_bbox_crop__lpips,sd24_0001,sd24_0001,sd__can__p001__loss_large__p00_generic__s2026_...,sd__can__p001__loss_large__p00_generic__s2026_...,p001_loss_large,p001_loss_large,...,generic_restoration,generic,0,False,model_inference,cuda,pipeline_default,30,7.5,1.0
3,sd_lpips_full_20260808T071919Z,945,2724,sd24_0002__full_image__lpips,sd24_0002,sd24_0002,sd__can__p001__loss_small__p00_generic__s2026_...,sd__can__p001__loss_small__p00_generic__s2026_...,p001_loss_small,p001_loss_small,...,generic_restoration,generic,0,False,model_inference,cuda,pipeline_default,30,7.5,1.0
4,sd_lpips_full_20260808T071919Z,945,2724,sd24_0002__content_region__lpips,sd24_0002,sd24_0002,sd__can__p001__loss_small__p00_generic__s2026_...,sd__can__p001__loss_small__p00_generic__s2026_...,p001_loss_small,p001_loss_small,...,generic_restoration,generic,0,False,model_inference,cuda,pipeline_default,30,7.5,1.0


In [16]:
# Batch 3 / Cell 16 - Validate full LPIPS metrics and update stage manifest
def batch3_validation_row(check_name: str, observed, expected, passed: bool, failure_message: str = "") -> dict[str, Any]:
    return {
        "check_name": check_name,
        "observed": observed,
        "expected": expected,
        "passed": bool(passed),
        "failure_message": "" if passed else failure_message,
    }


if not BATCH3_METRICS_PATH.is_file():
    raise FileNotFoundError(f"Full LPIPS metrics CSV not found: {BATCH3_METRICS_PATH}")

saved_full_lpips_metrics_df = pd.read_csv(BATCH3_METRICS_PATH)
saved_full_input_cases_df = pd.read_csv(BATCH1_INPUT_CASES_PATH)

ready_full_input_cases_df = saved_full_input_cases_df.loc[
    bool_series(saved_full_input_cases_df["batch1_ready_for_lpips"])
].copy()

expected_region_counts_from_saved = {
    "full_image": int(len(ready_full_input_cases_df)),
    "content_region": int(len(ready_full_input_cases_df)),
    "mask_bbox_crop": int(bool_series(ready_full_input_cases_df["mask_bbox_applicable"]).sum()),
}
expected_lpips_rows_from_saved = int(sum(expected_region_counts_from_saved.values()))

helper_full_validation_df = metrics_lpips.validate_lpips_metrics(
    saved_full_lpips_metrics_df,
    expected_rows=expected_lpips_rows_from_saved,
    expected_region_counts=expected_region_counts_from_saved,
)

helper_full_validation_normalized_df = helper_full_validation_df.rename(
    columns={
        "check": "check_name",
        "detail": "failure_message",
    }
).copy()
helper_full_validation_normalized_df["observed"] = ""
helper_full_validation_normalized_df["expected"] = ""
helper_full_validation_normalized_df["validation_source"] = "metrics_lpips.validate_lpips_metrics"

actual_region_counts = saved_full_lpips_metrics_df["evaluation_region"].value_counts().to_dict()
actual_region_counts = {str(key): int(value) for key, value in actual_region_counts.items()}

status_counts = saved_full_lpips_metrics_df["status"].value_counts(dropna=False).to_dict()
status_counts = {str(key): int(value) for key, value in status_counts.items()}

input_metric_case_ids = set(ready_full_input_cases_df["metric_case_id"].astype(str))
output_lpips_case_ids = set(saved_full_lpips_metrics_df["lpips_case_id"].dropna().astype(str))

missing_case_ids = sorted(input_metric_case_ids - output_lpips_case_ids)
extra_case_ids = sorted(output_lpips_case_ids - input_metric_case_ids)

expected_rows_by_case_df = ready_full_input_cases_df[
    ["metric_case_id", "expected_lpips_region_count"]
].copy()
expected_rows_by_case_df["metric_case_id"] = expected_rows_by_case_df["metric_case_id"].astype(str)

actual_rows_by_case_df = (
    saved_full_lpips_metrics_df
    .groupby(saved_full_lpips_metrics_df["lpips_case_id"].astype(str), dropna=False)
    .size()
    .rename("actual_lpips_region_count")
    .reset_index()
    .rename(columns={"lpips_case_id": "metric_case_id"})
)

case_region_count_check_df = expected_rows_by_case_df.merge(
    actual_rows_by_case_df,
    on="metric_case_id",
    how="left",
)
case_region_count_check_df["actual_lpips_region_count"] = (
    case_region_count_check_df["actual_lpips_region_count"].fillna(0).astype(int)
)
case_region_count_check_df["case_region_count_matches"] = (
    case_region_count_check_df["expected_lpips_region_count"].astype(int)
    == case_region_count_check_df["actual_lpips_region_count"].astype(int)
)

case_region_count_mismatches = case_region_count_check_df.loc[
    ~case_region_count_check_df["case_region_count_matches"],
    ["metric_case_id", "expected_lpips_region_count", "actual_lpips_region_count"],
].to_dict("records")

zero_control_case_ids = set(
    ready_full_input_cases_df.loc[
        bool_series(ready_full_input_cases_df["is_zero_control"]),
        "metric_case_id",
    ].astype(str)
)

zero_control_mask_bbox_rows = int(
    (
        saved_full_lpips_metrics_df["lpips_case_id"].astype(str).isin(zero_control_case_ids)
        & saved_full_lpips_metrics_df["evaluation_region"].astype(str).eq("mask_bbox_crop")
    ).sum()
)

ok_rows_df = saved_full_lpips_metrics_df.loc[
    saved_full_lpips_metrics_df["status"].astype(str).eq("ok")
].copy()

numeric_columns = ["damaged_lpips", "restored_lpips", "lpips_improvement"]
finite_numeric_values = bool(
    np.isfinite(
        ok_rows_df[numeric_columns].apply(pd.to_numeric, errors="coerce").to_numpy(dtype=float)
    ).all()
) if not ok_rows_df.empty else False

manual_batch3_validation_df = pd.DataFrame(
    [
        batch3_validation_row(
            "batch2_validation_passed",
            rel(BATCH2_VALIDATION_PATH),
            "all checks passed",
            BATCH2_VALIDATION_PATH.is_file()
            and bool_series(pd.read_csv(BATCH2_VALIDATION_PATH)["passed"]).all(),
            "Batch 2 validation is missing or did not pass.",
        ),
        batch3_validation_row(
            "full_metrics_written",
            rel(BATCH3_METRICS_PATH),
            "file exists",
            BATCH3_METRICS_PATH.is_file(),
            "Full LPIPS metrics CSV was not written.",
        ),
        batch3_validation_row(
            "input_case_count",
            len(ready_full_input_cases_df),
            EXPECTED_CANDIDATE_ROWS,
            len(ready_full_input_cases_df) == EXPECTED_CANDIDATE_ROWS,
            "Ready LPIPS input case count does not match the Notebook 24 contract.",
        ),
        batch3_validation_row(
            "full_row_count",
            len(saved_full_lpips_metrics_df),
            expected_lpips_rows_from_saved,
            len(saved_full_lpips_metrics_df) == expected_lpips_rows_from_saved,
            "Full LPIPS row count does not match selected-case region expectations.",
        ),
        batch3_validation_row(
            "all_input_cases_present",
            {"missing_case_ids": missing_case_ids[:20], "extra_case_ids": extra_case_ids[:20]},
            {"missing_case_ids": [], "extra_case_ids": []},
            len(missing_case_ids) == 0 and len(extra_case_ids) == 0,
            "Full LPIPS output case IDs do not exactly match Batch 1 ready input cases.",
        ),
        batch3_validation_row(
            "per_case_region_counts_match",
            case_region_count_mismatches[:20],
            [],
            len(case_region_count_mismatches) == 0,
            "One or more cases produced the wrong number of LPIPS region rows.",
        ),
        batch3_validation_row(
            "region_counts",
            actual_region_counts,
            expected_region_counts_from_saved,
            actual_region_counts == expected_region_counts_from_saved,
            "Full LPIPS region counts do not match expectations.",
        ),
        batch3_validation_row(
            "status_counts",
            status_counts,
            {"ok": expected_lpips_rows_from_saved},
            status_counts == {"ok": expected_lpips_rows_from_saved},
            "Full LPIPS produced non-ok rows.",
        ),
        batch3_validation_row(
            "zero_control_has_no_mask_bbox_crop",
            zero_control_mask_bbox_rows,
            0,
            zero_control_mask_bbox_rows == 0,
            "Zero-control cases produced mask_bbox_crop rows.",
        ),
        batch3_validation_row(
            "finite_full_lpips_values",
            finite_numeric_values,
            True,
            finite_numeric_values,
            "One or more successful full LPIPS values is non-finite.",
        ),
    ]
)

manual_batch3_validation_df["validation_source"] = "notebook_batch3_manual_checks"

batch3_validation_df = pd.concat(
    [
        manual_batch3_validation_df,
        helper_full_validation_normalized_df[
            ["check_name", "observed", "expected", "passed", "failure_message", "validation_source"]
        ],
    ],
    ignore_index=True,
)

batch3_validation_df.to_csv(BATCH3_VALIDATION_PATH, index=False)

batch3_passed = bool_series(batch3_validation_df["passed"]).all()

stage_manifest = json.loads(STAGE_MANIFEST_PATH.read_text(encoding="utf-8"))

stage_manifest["stage"] = "batch3_full_lpips_computation"
stage_manifest["stage_status"] = "passed" if batch3_passed else "failed"
stage_manifest["updated_at_utc"] = utc_now_iso()
stage_manifest["batch3"] = {
    "status": "passed" if batch3_passed else "failed",
    "metrics": rel(BATCH3_METRICS_PATH),
    "validation": rel(BATCH3_VALIDATION_PATH),
    "full_lpips_run_id": (
        saved_full_lpips_metrics_df["full_lpips_run_id"].iloc[0]
        if "full_lpips_run_id" in saved_full_lpips_metrics_df.columns and len(saved_full_lpips_metrics_df)
        else ""
    ),
    "input_case_count": int(len(ready_full_input_cases_df)),
    "expected_lpips_rows": int(expected_lpips_rows_from_saved),
    "actual_lpips_rows": int(len(saved_full_lpips_metrics_df)),
    "expected_region_counts": expected_region_counts_from_saved,
    "actual_region_counts": actual_region_counts,
    "status_counts": status_counts,
    "lpips_net": LPIPS_NET,
    "lpips_input_size": int(LPIPS_INPUT_SIZE),
    "mask_bbox_margin": int(MASK_BBOX_MARGIN),
    "mask_binary_threshold": int(MASK_BINARY_THRESHOLD),
    "target_size": int(TARGET_SIZE),
    "device": str(globals().get("lpips_device", "")),
    "model_load_seconds": float(globals().get("full_lpips_model_load_seconds", np.nan)),
    "full_runtime_seconds": float(globals().get("full_lpips_runtime_seconds", np.nan)),
    "force_recompute": bool(FORCE_RECOMPUTE_FULL_LPIPS),
    "checks_passed": int(bool_series(batch3_validation_df["passed"]).sum()),
    "checks_total": int(len(batch3_validation_df)),
}

STAGE_MANIFEST_PATH.write_text(json.dumps(to_json_safe(stage_manifest), indent=2), encoding="utf-8")

print(f"Saved: {rel(BATCH3_VALIDATION_PATH)}")
print(f"Updated: {rel(STAGE_MANIFEST_PATH)}")
print(f"Batch 3 checks passed: {int(bool_series(batch3_validation_df['passed']).sum())} / {len(batch3_validation_df)}")

display(batch3_validation_df)

if not batch3_passed:
    display(batch3_validation_df.loc[~bool_series(batch3_validation_df["passed"]), ["check_name", "failure_message"]])

    if status_counts != {"ok": expected_lpips_rows_from_saved}:
        display(
            saved_full_lpips_metrics_df.loc[
                saved_full_lpips_metrics_df["status"].astype(str).ne("ok"),
                [
                    "lpips_case_id",
                    "candidate_id",
                    "case_id",
                    "mask_type",
                    "evaluation_region",
                    "status",
                    "issue",
                ],
            ].head(50)
        )

    raise RuntimeError("Batch 3 validation failed.")

Saved: outputs/24_stable_diffusion_lpips_metrics/validation/stable_diffusion_lpips_metrics_validation.csv
Updated: outputs/24_stable_diffusion_lpips_metrics/manifests/stable_diffusion_lpips_metrics_stage_manifest.json
Batch 3 checks passed: 18 / 18


,check_name,observed,expected,passed,failure_message,validation_source
0,batch2_validation_passed,outputs/24_stable_diffusion_lpips_metrics/vali...,all checks passed,True,,notebook_batch3_manual_checks
1,full_metrics_written,outputs/24_stable_diffusion_lpips_metrics/metr...,file exists,True,,notebook_batch3_manual_checks
2,input_case_count,945,945,True,,notebook_batch3_manual_checks
3,full_row_count,2724,2724,True,,notebook_batch3_manual_checks
4,all_input_cases_present,"{'missing_case_ids': [], 'extra_case_ids': []}","{'missing_case_ids': [], 'extra_case_ids': []}",True,,notebook_batch3_manual_checks
5,per_case_region_counts_match,[],[],True,,notebook_batch3_manual_checks
6,region_counts,"{'full_image': 945, 'content_region': 945, 'ma...","{'full_image': 945, 'content_region': 945, 'ma...",True,,notebook_batch3_manual_checks
7,status_counts,{'ok': 2724},{'ok': 2724},True,,notebook_batch3_manual_checks
8,zero_control_has_no_mask_bbox_crop,0,0,True,,notebook_batch3_manual_checks
9,finite_full_lpips_values,True,True,True,,notebook_batch3_manual_checks


In [17]:
# Batch 4 / Cell 17 - Build compact LPIPS summaries
if not BATCH3_VALIDATION_PATH.is_file():
    raise FileNotFoundError(f"Batch 3 validation not found: {BATCH3_VALIDATION_PATH}")

batch3_validation_df = pd.read_csv(BATCH3_VALIDATION_PATH)
if not bool_series(batch3_validation_df["passed"]).all():
    display(batch3_validation_df.loc[~bool_series(batch3_validation_df["passed"])])
    raise RuntimeError("Batch 3 did not pass. Do not continue to Batch 4.")

if not BATCH3_METRICS_PATH.is_file():
    raise FileNotFoundError(f"Batch 3 metrics not found: {BATCH3_METRICS_PATH}")

batch4_lpips_metrics_df = pd.read_csv(BATCH3_METRICS_PATH)

if batch4_lpips_metrics_df.empty:
    raise RuntimeError("Batch 3 LPIPS metrics CSV is empty.")

ok_lpips_metrics_df = batch4_lpips_metrics_df.loc[
    batch4_lpips_metrics_df["status"].astype(str).eq("ok")
].copy()

if ok_lpips_metrics_df.empty:
    raise RuntimeError("No successful LPIPS rows available for Batch 4 summaries.")

summary_specs = [
    ("overall", []),
    ("by_evaluation_region", ["evaluation_region"]),
    ("by_dataset_region", ["dataset_name", "evaluation_region"]),
    ("by_mask_type_region", ["mask_type", "evaluation_region"]),
    ("by_zero_control_region", ["is_zero_control", "evaluation_region"]),
    ("by_category_region", ["category", "evaluation_region"]),
    ("by_prompt_family_region", ["prompt_variant_family", "evaluation_region"]),
]

summary_frames = []
expected_summary_scopes = []

for summary_scope, group_columns in summary_specs:
    available_group_columns = [
        column for column in group_columns
        if column in ok_lpips_metrics_df.columns
    ]

    if len(available_group_columns) != len(group_columns):
        print(f"Skipping {summary_scope}; missing columns: {sorted(set(group_columns) - set(available_group_columns))}")
        continue

    scope_summary_df = metrics_lpips.summarize_lpips_metrics(
        ok_lpips_metrics_df,
        group_columns=available_group_columns,
        summary_scope=summary_scope,
    )
    scope_summary_df.insert(1, "group_columns", " | ".join(available_group_columns) if available_group_columns else "none")
    summary_frames.append(scope_summary_df)
    expected_summary_scopes.append(summary_scope)

batch4_summary_df = pd.concat(summary_frames, ignore_index=True, sort=False)

front_summary_columns = [
    "summary_scope",
    "group_columns",
    "dataset_name",
    "mask_type",
    "category",
    "prompt_variant_family",
    "is_zero_control",
    "evaluation_region",
    "rows",
    "cases",
    "median_damaged_lpips",
    "median_restored_lpips",
    "median_lpips_improvement",
    "mean_damaged_lpips",
    "mean_restored_lpips",
    "mean_lpips_improvement",
    "improvement_rate",
    "median_region_pixel_count",
]
front_summary_columns = [column for column in front_summary_columns if column in batch4_summary_df.columns]
remaining_summary_columns = [column for column in batch4_summary_df.columns if column not in front_summary_columns]
batch4_summary_df = batch4_summary_df[front_summary_columns + remaining_summary_columns].copy()

batch4_summary_df.to_csv(BATCH4_SUMMARY_PATH, index=False)

print(f"Saved: {rel(BATCH4_SUMMARY_PATH)}")
print(f"Summary rows: {len(batch4_summary_df):,}")
print(f"Summary scopes: {expected_summary_scopes}")

display(
    batch4_summary_df.loc[
        batch4_summary_df["summary_scope"].isin(["overall", "by_evaluation_region", "by_zero_control_region"])
    ]
)

Saved: outputs/24_stable_diffusion_lpips_metrics/analysis/stable_diffusion_lpips_summary.csv
Summary rows: 83
Summary scopes: ['overall', 'by_evaluation_region', 'by_dataset_region', 'by_mask_type_region', 'by_zero_control_region', 'by_category_region', 'by_prompt_family_region']


,summary_scope,group_columns,dataset_name,mask_type,category,prompt_variant_family,is_zero_control,evaluation_region,rows,cases,median_damaged_lpips,median_restored_lpips,median_lpips_improvement,mean_damaged_lpips,mean_restored_lpips,mean_lpips_improvement,improvement_rate,median_region_pixel_count
0,overall,none,NaN,NaN,NaN,NaN,NaN,NaN,2724,945,0.154168,0.071574,0.093542,0.178591,0.109773,0.068818,0.616740,509184.0
1,by_evaluation_region,evaluation_region,NaN,NaN,NaN,NaN,NaN,full_image,945,945,0.121364,0.051201,0.072054,0.130986,0.072892,0.058093,0.592593,589824.0
2,by_evaluation_region,evaluation_region,NaN,NaN,NaN,NaN,NaN,content_region,945,945,0.155152,0.069245,0.083356,0.154951,0.095241,0.059710,0.592593,436992.0
3,by_evaluation_region,evaluation_region,NaN,NaN,NaN,NaN,NaN,mask_bbox_crop,834,834,0.212747,0.133162,0.159018,0.259318,0.168027,0.091291,0.671463,370881.0
57,by_zero_control_region,is_zero_control | evaluation_region,NaN,NaN,NaN,NaN,False,full_image,851,851,0.134243,0.060296,0.085130,0.145454,0.080944,0.064510,0.658049,589824.0
58,by_zero_control_region,is_zero_control | evaluation_region,NaN,NaN,NaN,NaN,False,content_region,851,851,0.167880,0.081384,0.103330,0.172066,0.105761,0.066306,0.658049,434688.0
59,by_zero_control_region,is_zero_control | evaluation_region,NaN,NaN,NaN,NaN,False,mask_bbox_crop,834,834,0.212747,0.133162,0.159018,0.259318,0.168027,0.091291,0.671463,370881.0
60,by_zero_control_region,is_zero_control | evaluation_region,NaN,NaN,NaN,NaN,True,full_image,94,94,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,589824.0
61,by_zero_control_region,is_zero_control | evaluation_region,NaN,NaN,NaN,NaN,True,content_region,94,94,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,452352.0


In [18]:
# Batch 4 / Cell 18 - Rank and select representative LPIPS cases
if not BATCH1_INPUT_CASES_PATH.is_file():
    raise FileNotFoundError(f"Batch 1 input cases not found: {BATCH1_INPUT_CASES_PATH}")

batch4_input_cases_df = pd.read_csv(BATCH1_INPUT_CASES_PATH)

ranked_lpips_df = ok_lpips_metrics_df.copy()
ranked_lpips_df["lpips_improvement"] = pd.to_numeric(ranked_lpips_df["lpips_improvement"], errors="coerce")
ranked_lpips_df["damaged_lpips"] = pd.to_numeric(ranked_lpips_df["damaged_lpips"], errors="coerce")
ranked_lpips_df["restored_lpips"] = pd.to_numeric(ranked_lpips_df["restored_lpips"], errors="coerce")
ranked_lpips_df["absolute_lpips_improvement"] = ranked_lpips_df["lpips_improvement"].abs()

ranked_lpips_df["quality_direction"] = np.select(
    [
        ranked_lpips_df["lpips_improvement"] > 0,
        ranked_lpips_df["lpips_improvement"] < 0,
    ],
    [
        "improved",
        "regressed",
    ],
    default="unchanged",
)

ranked_lpips_df["lpips_improvement_rank_desc_in_region"] = (
    ranked_lpips_df.groupby("evaluation_region")["lpips_improvement"]
    .rank(method="first", ascending=False)
    .astype(int)
)
ranked_lpips_df["lpips_improvement_rank_asc_in_region"] = (
    ranked_lpips_df.groupby("evaluation_region")["lpips_improvement"]
    .rank(method="first", ascending=True)
    .astype(int)
)
ranked_lpips_df["restored_lpips_rank_asc_in_region"] = (
    ranked_lpips_df.groupby("evaluation_region")["restored_lpips"]
    .rank(method="first", ascending=True)
    .astype(int)
)

SELECTED_TOP_N_PER_GROUP = 8
selected_frames = []

for evaluation_region in ["mask_bbox_crop", "content_region", "full_image"]:
    region_df = ranked_lpips_df.loc[
        ranked_lpips_df["evaluation_region"].astype(str).eq(evaluation_region)
    ].copy()

    if region_df.empty:
        print(f"Skipping selected cases for missing region: {evaluation_region}")
        continue

    strongest_df, weakest_df = metrics_lpips.rank_lpips_cases(
        ranked_lpips_df,
        evaluation_region=evaluation_region,
        top_n=SELECTED_TOP_N_PER_GROUP,
    )

    strongest_df = strongest_df.copy()
    strongest_df.insert(0, "selection_group", f"strongest_improvement__{evaluation_region}")
    strongest_df.insert(1, "selection_reason", f"Highest LPIPS improvement in {evaluation_region}.")
    strongest_df.insert(2, "selection_rank", np.arange(1, len(strongest_df) + 1))

    weakest_df = weakest_df.copy()
    weakest_df.insert(0, "selection_group", f"weakest_or_regressed__{evaluation_region}")
    weakest_df.insert(1, "selection_reason", f"Lowest LPIPS improvement in {evaluation_region}.")
    weakest_df.insert(2, "selection_rank", np.arange(1, len(weakest_df) + 1))

    selected_frames.extend([strongest_df, weakest_df])

zero_control_df = ranked_lpips_df.loc[
    bool_series(ranked_lpips_df["is_zero_control"])
    & ranked_lpips_df["evaluation_region"].astype(str).isin(["full_image", "content_region"])
].copy()

if not zero_control_df.empty:
    zero_control_worst_df = (
        zero_control_df
        .sort_values(["lpips_improvement", "evaluation_region", "lpips_case_id"], ascending=[True, True, True], kind="stable")
        .head(SELECTED_TOP_N_PER_GROUP)
        .copy()
    )
    zero_control_worst_df.insert(0, "selection_group", "zero_control_largest_regression")
    zero_control_worst_df.insert(1, "selection_reason", "Zero-control rows with the largest restoration-side LPIPS regression.")
    zero_control_worst_df.insert(2, "selection_rank", np.arange(1, len(zero_control_worst_df) + 1))
    selected_frames.append(zero_control_worst_df)

batch4_selected_cases_df = pd.concat(selected_frames, ignore_index=True, sort=False)
batch4_selected_cases_df["selected_case_key"] = (
    batch4_selected_cases_df["selection_group"].astype(str)
    + "__"
    + batch4_selected_cases_df["lpips_row_id"].astype(str)
)

input_join_columns = [
    "metric_case_id",
    "clean_path",
    "damaged_path",
    "restored_path",
    "mask_path",
    "content_x_min",
    "content_y_min",
    "content_x_max",
    "content_y_max",
    "content_box_source",
    "mask_bbox_applicable",
    "expected_lpips_region_count",
]
input_join_columns = [
    column for column in input_join_columns
    if column in batch4_input_cases_df.columns
]

batch4_selected_cases_df = batch4_selected_cases_df.merge(
    batch4_input_cases_df[input_join_columns].drop_duplicates("metric_case_id"),
    on="metric_case_id",
    how="left",
    suffixes=("", "_input"),
)

front_selected_columns = [
    "selected_case_key",
    "selection_group",
    "selection_reason",
    "selection_rank",
    "lpips_row_id",
    "lpips_case_id",
    "metric_case_id",
    "candidate_id",
    "candidate_index",
    "candidate_seed",
    "effective_candidate_seed",
    "case_id",
    "source_case_id",
    "restoration_case_id",
    "dataset_name",
    "metric_applicability",
    "painting_id",
    "category",
    "title",
    "artist",
    "style",
    "model_name",
    "mask_id",
    "mask_type",
    "is_zero_control",
    "prompt_policy_id",
    "prompt_variant_id",
    "prompt_template_name",
    "prompt_variant_family",
    "evaluation_region",
    "region_x_min",
    "region_y_min",
    "region_x_max",
    "region_y_max",
    "region_pixel_count",
    "mask_area_pixels",
    "damaged_lpips",
    "restored_lpips",
    "lpips_improvement",
    "absolute_lpips_improvement",
    "quality_direction",
    "lpips_improvement_rank_desc_in_region",
    "lpips_improvement_rank_asc_in_region",
    "restored_lpips_rank_asc_in_region",
    "clean_path",
    "damaged_path",
    "restored_path",
    "mask_path",
    "content_x_min",
    "content_y_min",
    "content_x_max",
    "content_y_max",
    "content_box_source",
    "mask_bbox_applicable",
    "expected_lpips_region_count",
]
front_selected_columns = [column for column in front_selected_columns if column in batch4_selected_cases_df.columns]
remaining_selected_columns = [column for column in batch4_selected_cases_df.columns if column not in front_selected_columns]

batch4_selected_cases_df = batch4_selected_cases_df[front_selected_columns + remaining_selected_columns].copy()
batch4_selected_cases_df.to_csv(BATCH4_SELECTED_CASES_PATH, index=False)

print(f"Saved: {rel(BATCH4_SELECTED_CASES_PATH)}")
print(f"Selected rows: {len(batch4_selected_cases_df):,}")

display(
    batch4_selected_cases_df[
        [
            "selection_group",
            "selection_rank",
            "candidate_id",
            "mask_type",
            "evaluation_region",
            "damaged_lpips",
            "restored_lpips",
            "lpips_improvement",
            "quality_direction",
        ]
    ].head(30)
)

Saved: outputs/24_stable_diffusion_lpips_metrics/analysis/stable_diffusion_lpips_selected_cases.csv
Selected rows: 56


,selection_group,selection_rank,candidate_id,mask_type,evaluation_region,damaged_lpips,restored_lpips,lpips_improvement,quality_direction
0,strongest_improvement__mask_bbox_crop,1,sd__dsz__p001__loss_large__p00_generic__s2026_...,loss_large,mask_bbox_crop,0.609372,0.114338,0.495033,improved
1,strongest_improvement__mask_bbox_crop,2,sd__mrob__p001__loss_large__p00_generic__s2026...,loss_large,mask_bbox_crop,0.721254,0.229863,0.491390,improved
2,strongest_improvement__mask_bbox_crop,3,sd__dsz__p001__loss_large__p00_generic__s2026_...,loss_large,mask_bbox_crop,0.654634,0.164102,0.490533,improved
3,strongest_improvement__mask_bbox_crop,4,sd__dsz__p001__loss_large__p01_style_period__s...,loss_large,mask_bbox_crop,0.654634,0.167671,0.486964,improved
4,strongest_improvement__mask_bbox_crop,5,sd__can__p001__loss_large__p00_generic__s2026_...,loss_large,mask_bbox_crop,0.637281,0.150479,0.486802,improved
5,strongest_improvement__mask_bbox_crop,6,sd__dsz__p001__loss_large__p04_full_context__s...,loss_large,mask_bbox_crop,0.654634,0.168145,0.486490,improved
6,strongest_improvement__mask_bbox_crop,7,sd__can__p032__loss_large__p00_generic__s2026_...,loss_large,mask_bbox_crop,0.541988,0.060104,0.481884,improved
7,strongest_improvement__mask_bbox_crop,8,sd__dsz__p001__loss_large__p00_generic__s2026_...,loss_large,mask_bbox_crop,0.646574,0.168396,0.478179,improved
8,weakest_or_regressed__mask_bbox_crop,1,sd__syn__p001__dirt_dust__p04_full_context__s2...,dirt_dust,mask_bbox_crop,0.000256,0.498590,-0.498334,regressed
9,weakest_or_regressed__mask_bbox_crop,2,sd__syn__p001__dirt_dust__p03_artist_style_p__...,dirt_dust,mask_bbox_crop,0.000256,0.498173,-0.497917,regressed


In [19]:
# Batch 4 / Cell 19 - Validate Batch 4 analysis outputs and update stage manifest
def batch4_validation_row(check_name: str, observed, expected, passed: bool, failure_message: str = "") -> dict[str, Any]:
    return {
        "check_name": check_name,
        "observed": observed,
        "expected": expected,
        "passed": bool(passed),
        "failure_message": "" if passed else failure_message,
    }


saved_batch4_summary_df = pd.read_csv(BATCH4_SUMMARY_PATH) if BATCH4_SUMMARY_PATH.is_file() else pd.DataFrame()
saved_batch4_selected_cases_df = pd.read_csv(BATCH4_SELECTED_CASES_PATH) if BATCH4_SELECTED_CASES_PATH.is_file() else pd.DataFrame()
saved_batch3_metrics_df = pd.read_csv(BATCH3_METRICS_PATH)

summary_scopes_observed = (
    sorted(saved_batch4_summary_df["summary_scope"].dropna().astype(str).unique())
    if "summary_scope" in saved_batch4_summary_df.columns
    else []
)

selected_groups_observed = (
    sorted(saved_batch4_selected_cases_df["selection_group"].dropna().astype(str).unique())
    if "selection_group" in saved_batch4_selected_cases_df.columns
    else []
)

expected_selected_groups = []
for evaluation_region in ["mask_bbox_crop", "content_region", "full_image"]:
    if evaluation_region in set(saved_batch3_metrics_df["evaluation_region"].astype(str)):
        expected_selected_groups.extend(
            [
                f"strongest_improvement__{evaluation_region}",
                f"weakest_or_regressed__{evaluation_region}",
            ]
        )

if bool_series(saved_batch3_metrics_df.get("is_zero_control", pd.Series([], dtype=bool))).any():
    expected_selected_groups.append("zero_control_largest_regression")

summary_numeric_columns = [
    column for column in [
        "rows",
        "cases",
        "median_damaged_lpips",
        "median_restored_lpips",
        "median_lpips_improvement",
        "mean_damaged_lpips",
        "mean_restored_lpips",
        "mean_lpips_improvement",
        "improvement_rate",
        "median_region_pixel_count",
    ]
    if column in saved_batch4_summary_df.columns
]

summary_numeric_finite = bool(
    np.isfinite(
        saved_batch4_summary_df[summary_numeric_columns]
        .apply(pd.to_numeric, errors="coerce")
        .to_numpy(dtype=float)
    ).all()
) if summary_numeric_columns and not saved_batch4_summary_df.empty else False

selected_metric_case_ids = (
    set(saved_batch4_selected_cases_df["metric_case_id"].dropna().astype(str))
    if "metric_case_id" in saved_batch4_selected_cases_df.columns
    else set()
)
metric_case_ids = set(saved_batch3_metrics_df["metric_case_id"].dropna().astype(str))
unknown_selected_case_ids = sorted(selected_metric_case_ids - metric_case_ids)

selected_duplicate_keys = (
    int(saved_batch4_selected_cases_df.duplicated(["selected_case_key"], keep=False).sum())
    if "selected_case_key" in saved_batch4_selected_cases_df.columns
    else len(saved_batch4_selected_cases_df)
)

required_selected_columns = [
    "selected_case_key",
    "selection_group",
    "selection_reason",
    "selection_rank",
    "metric_case_id",
    "candidate_id",
    "evaluation_region",
    "damaged_lpips",
    "restored_lpips",
    "lpips_improvement",
    "quality_direction",
    "clean_path",
    "damaged_path",
    "restored_path",
    "mask_path",
]
missing_selected_columns = [
    column for column in required_selected_columns
    if column not in saved_batch4_selected_cases_df.columns
]

path_missing_counts = {}
if not saved_batch4_selected_cases_df.empty:
    for path_column in ["clean_path", "damaged_path", "restored_path", "mask_path"]:
        if path_column in saved_batch4_selected_cases_df.columns:
            path_missing_counts[path_column] = int(
                saved_batch4_selected_cases_df[path_column]
                .astype(str)
                .map(lambda value: not resolve_project_path(value).is_file())
                .sum()
            )

batch4_validation_df = pd.DataFrame(
    [
        batch4_validation_row(
            "batch3_validation_passed",
            rel(BATCH3_VALIDATION_PATH),
            "all checks passed",
            BATCH3_VALIDATION_PATH.is_file()
            and bool_series(pd.read_csv(BATCH3_VALIDATION_PATH)["passed"]).all(),
            "Batch 3 validation is missing or did not pass.",
        ),
        batch4_validation_row(
            "summary_written",
            rel(BATCH4_SUMMARY_PATH),
            "file exists",
            BATCH4_SUMMARY_PATH.is_file(),
            "Batch 4 summary CSV was not written.",
        ),
        batch4_validation_row(
            "selected_cases_written",
            rel(BATCH4_SELECTED_CASES_PATH),
            "file exists",
            BATCH4_SELECTED_CASES_PATH.is_file(),
            "Batch 4 selected cases CSV was not written.",
        ),
        batch4_validation_row(
            "summary_non_empty",
            len(saved_batch4_summary_df),
            "> 0",
            len(saved_batch4_summary_df) > 0,
            "Batch 4 summary CSV is empty.",
        ),
        batch4_validation_row(
            "selected_cases_non_empty",
            len(saved_batch4_selected_cases_df),
            "> 0",
            len(saved_batch4_selected_cases_df) > 0,
            "Batch 4 selected cases CSV is empty.",
        ),
        batch4_validation_row(
            "expected_summary_scopes_present",
            summary_scopes_observed,
            expected_summary_scopes,
            set(expected_summary_scopes).issubset(set(summary_scopes_observed)),
            "One or more expected summary scopes is missing.",
        ),
        batch4_validation_row(
            "summary_numeric_values_finite",
            summary_numeric_finite,
            True,
            summary_numeric_finite,
            "One or more numeric summary values is non-finite.",
        ),
        batch4_validation_row(
            "required_selected_columns_present",
            missing_selected_columns,
            [],
            len(missing_selected_columns) == 0,
            "Selected cases CSV is missing required columns.",
        ),
        batch4_validation_row(
            "expected_selected_groups_present",
            selected_groups_observed,
            expected_selected_groups,
            set(expected_selected_groups).issubset(set(selected_groups_observed)),
            "One or more expected selected-case groups is missing.",
        ),
        batch4_validation_row(
            "selected_case_keys_unique",
            selected_duplicate_keys,
            0,
            selected_duplicate_keys == 0,
            "Selected case keys are not unique.",
        ),
        batch4_validation_row(
            "selected_cases_map_to_metrics",
            unknown_selected_case_ids[:20],
            [],
            len(unknown_selected_case_ids) == 0,
            "One or more selected cases does not map back to Batch 3 metrics.",
        ),
        batch4_validation_row(
            "selected_case_paths_exist",
            path_missing_counts,
            {"clean_path": 0, "damaged_path": 0, "restored_path": 0, "mask_path": 0},
            all(value == 0 for value in path_missing_counts.values()),
            "One or more selected-case image paths is missing.",
        ),
    ]
)

batch4_validation_df.to_csv(BATCH4_VALIDATION_PATH, index=False)
batch4_passed = bool_series(batch4_validation_df["passed"]).all()

stage_manifest = json.loads(STAGE_MANIFEST_PATH.read_text(encoding="utf-8"))

stage_manifest["stage"] = "batch4_compact_summaries_selected_cases"
stage_manifest["stage_status"] = "passed" if batch4_passed else "failed"
stage_manifest["updated_at_utc"] = utc_now_iso()
stage_manifest["batch4"] = {
    "status": "passed" if batch4_passed else "failed",
    "summary": rel(BATCH4_SUMMARY_PATH),
    "selected_cases": rel(BATCH4_SELECTED_CASES_PATH),
    "validation": rel(BATCH4_VALIDATION_PATH),
    "summary_rows": int(len(saved_batch4_summary_df)),
    "selected_case_rows": int(len(saved_batch4_selected_cases_df)),
    "summary_scopes": summary_scopes_observed,
    "selected_groups": selected_groups_observed,
    "checks_passed": int(bool_series(batch4_validation_df["passed"]).sum()),
    "checks_total": int(len(batch4_validation_df)),
}

STAGE_MANIFEST_PATH.write_text(json.dumps(to_json_safe(stage_manifest), indent=2), encoding="utf-8")

print(f"Saved: {rel(BATCH4_VALIDATION_PATH)}")
print(f"Updated: {rel(STAGE_MANIFEST_PATH)}")
print(f"Batch 4 checks passed: {int(bool_series(batch4_validation_df['passed']).sum())} / {len(batch4_validation_df)}")

display(batch4_validation_df)

if not batch4_passed:
    display(batch4_validation_df.loc[~bool_series(batch4_validation_df["passed"]), ["check_name", "failure_message"]])
    raise RuntimeError("Batch 4 validation failed.")

Saved: outputs/24_stable_diffusion_lpips_metrics/validation/batch4_analysis_validation.csv
Updated: outputs/24_stable_diffusion_lpips_metrics/manifests/stable_diffusion_lpips_metrics_stage_manifest.json
Batch 4 checks passed: 12 / 12


,check_name,observed,expected,passed,failure_message
0,batch3_validation_passed,outputs/24_stable_diffusion_lpips_metrics/vali...,all checks passed,True,
1,summary_written,outputs/24_stable_diffusion_lpips_metrics/anal...,file exists,True,
2,selected_cases_written,outputs/24_stable_diffusion_lpips_metrics/anal...,file exists,True,
3,summary_non_empty,83,> 0,True,
4,selected_cases_non_empty,56,> 0,True,
5,expected_summary_scopes_present,"[by_category_region, by_dataset_region, by_eva...","[overall, by_evaluation_region, by_dataset_reg...",True,
6,summary_numeric_values_finite,True,True,True,
7,required_selected_columns_present,[],[],True,
8,expected_selected_groups_present,"[strongest_improvement__content_region, strong...","[strongest_improvement__mask_bbox_crop, weakes...",True,
9,selected_case_keys_unique,0,0,True,


In [21]:
# Batch 5 / Cell 20 - Load validated Batch 4 outputs and define figure helpers
import re
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw, ImageOps

# Defensive path setup in case the notebook kernel was started before Batch 5 paths existed.
BATCH5_FIGURE_DIR = globals().get(
    "BATCH5_FIGURE_DIR",
    OUTPUT_DIRS["figures"] / "lpips_diagnostics",
)
BATCH5_FIGURE_MANIFEST_PATH = globals().get(
    "BATCH5_FIGURE_MANIFEST_PATH",
    OUTPUT_DIRS["figures"] / "stable_diffusion_lpips_figure_manifest.csv",
)
BATCH5_VALIDATION_PATH = globals().get(
    "BATCH5_VALIDATION_PATH",
    OUTPUT_DIRS["validation"] / "batch5_figure_validation.csv",
)

if not BATCH4_VALIDATION_PATH.is_file():
    raise FileNotFoundError(f"Batch 4 validation not found: {BATCH4_VALIDATION_PATH}")

batch4_validation_df = pd.read_csv(BATCH4_VALIDATION_PATH)
if not bool_series(batch4_validation_df["passed"]).all():
    display(batch4_validation_df.loc[~bool_series(batch4_validation_df["passed"])])
    raise RuntimeError("Batch 4 did not pass. Do not continue to Batch 5.")

for required_path in [BATCH3_METRICS_PATH, BATCH4_SUMMARY_PATH, BATCH4_SELECTED_CASES_PATH]:
    if not required_path.is_file():
        raise FileNotFoundError(f"Required Batch 5 input is missing: {required_path}")

BATCH5_FIGURE_DIR.mkdir(parents=True, exist_ok=True)

batch5_metrics_df = pd.read_csv(BATCH3_METRICS_PATH)
batch5_summary_df = pd.read_csv(BATCH4_SUMMARY_PATH)
batch5_selected_cases_df = pd.read_csv(BATCH4_SELECTED_CASES_PATH)

ok_batch5_metrics_df = batch5_metrics_df.loc[
    batch5_metrics_df["status"].astype(str).eq("ok")
].copy()

if ok_batch5_metrics_df.empty:
    raise RuntimeError("No successful LPIPS metrics available for Batch 5 figures.")

for numeric_column in ["damaged_lpips", "restored_lpips", "lpips_improvement", "mask_area_pixels"]:
    if numeric_column in ok_batch5_metrics_df.columns:
        ok_batch5_metrics_df[numeric_column] = pd.to_numeric(ok_batch5_metrics_df[numeric_column], errors="coerce")
    if numeric_column in batch5_selected_cases_df.columns:
        batch5_selected_cases_df[numeric_column] = pd.to_numeric(batch5_selected_cases_df[numeric_column], errors="coerce")

REGION_ORDER = [
    region for region in ["full_image", "content_region", "mask_bbox_crop"]
    if region in set(ok_batch5_metrics_df["evaluation_region"].astype(str))
]

REGION_LABELS = {
    "full_image": "Full image",
    "content_region": "Content region",
    "mask_bbox_crop": "Mask bbox crop",
}

FIGURE_DPI = 160
SELECTED_PANEL_ROWS_PER_GROUP = 2
PANEL_THUMB_SIZE = 256
BATCH5_CREATED_AT_UTC = utc_now_iso()

figure_records = []


def safe_slug(value: str, max_length: int = 120) -> str:
    slug = re.sub(r"[^A-Za-z0-9._-]+", "_", str(value).strip())
    slug = re.sub(r"_+", "_", slug).strip("_")
    return (slug or "item")[:max_length]


def resolve_image_path(path_value: str | Path) -> Path:
    raw = str(path_value).strip()
    direct_path = Path(raw)
    if direct_path.is_file():
        return direct_path

    project_path = resolve_project_path(raw)
    if project_path.is_file():
        return project_path

    normalized_project_path = PROJECT_ROOT / raw.replace("\\", "/")
    return normalized_project_path


def register_figure(
    figure_id: str,
    figure_type: str,
    title: str,
    description: str,
    output_path: Path,
    source_table: str,
    source_rows: int,
    **metadata,
) -> None:
    output_path = Path(output_path)
    record = {
        "figure_id": figure_id,
        "figure_type": figure_type,
        "title": title,
        "description": description,
        "path": str(output_path),
        "path_project_relative": rel(output_path),
        "exists": output_path.is_file(),
        "size_bytes": int(output_path.stat().st_size) if output_path.is_file() else 0,
        "source_table": source_table,
        "source_rows": int(source_rows),
        "created_at_utc": BATCH5_CREATED_AT_UTC,
    }
    record.update(metadata)
    figure_records.append(record)


def save_matplotlib_figure(
    fig,
    filename: str,
    figure_type: str,
    title: str,
    description: str,
    source_table: str,
    source_rows: int,
    **metadata,
) -> Path:
    output_path = BATCH5_FIGURE_DIR / filename
    fig.tight_layout()
    fig.savefig(output_path, dpi=FIGURE_DPI, bbox_inches="tight")
    plt.close(fig)

    register_figure(
        figure_id=Path(filename).stem,
        figure_type=figure_type,
        title=title,
        description=description,
        output_path=output_path,
        source_table=source_table,
        source_rows=source_rows,
        **metadata,
    )
    return output_path


def finite_plot_df(dataframe: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    plot_df = dataframe.copy()
    for column in columns:
        plot_df[column] = pd.to_numeric(plot_df[column], errors="coerce")
    return plot_df.replace([np.inf, -np.inf], np.nan).dropna(subset=columns).copy()


print(f"Batch 5 metrics rows: {len(ok_batch5_metrics_df):,}")
print(f"Batch 5 selected-case rows: {len(batch5_selected_cases_df):,}")
print(f"Figure directory: {rel(BATCH5_FIGURE_DIR)}")

Batch 5 metrics rows: 2,724
Batch 5 selected-case rows: 56
Figure directory: outputs/24_stable_diffusion_lpips_metrics/figures/lpips_diagnostics


In [22]:
# Batch 5 / Cell 21 - Generate aggregate LPIPS diagnostic plots
scatter_df = finite_plot_df(ok_batch5_metrics_df, ["damaged_lpips", "restored_lpips"])

fig, ax = plt.subplots(figsize=(7.2, 6.2))
for evaluation_region in REGION_ORDER:
    region_df = scatter_df.loc[scatter_df["evaluation_region"].astype(str).eq(evaluation_region)]
    ax.scatter(
        region_df["damaged_lpips"],
        region_df["restored_lpips"],
        s=14,
        alpha=0.45,
        label=f"{REGION_LABELS.get(evaluation_region, evaluation_region)} ({len(region_df):,})",
    )

axis_min = float(np.nanmin(scatter_df[["damaged_lpips", "restored_lpips"]].to_numpy()))
axis_max = float(np.nanmax(scatter_df[["damaged_lpips", "restored_lpips"]].to_numpy()))
ax.plot([axis_min, axis_max], [axis_min, axis_max], color="black", linewidth=1, linestyle="--")
ax.set_title("Stable Diffusion LPIPS: damaged versus restored")
ax.set_xlabel("Damaged LPIPS vs clean")
ax.set_ylabel("Restored LPIPS vs clean")
ax.legend(frameon=False, fontsize=8)
ax.grid(alpha=0.25)

save_matplotlib_figure(
    fig,
    "stable_diffusion_lpips_damaged_vs_restored_scatter.png",
    "aggregate_plot",
    "Damaged versus restored LPIPS",
    "Points below the diagonal indicate lower LPIPS after restoration for the evaluated region.",
    source_table=rel(BATCH3_METRICS_PATH),
    source_rows=len(scatter_df),
)

box_df = finite_plot_df(ok_batch5_metrics_df, ["lpips_improvement"])
box_values = [
    box_df.loc[box_df["evaluation_region"].astype(str).eq(region), "lpips_improvement"].to_numpy()
    for region in REGION_ORDER
]
box_labels = [REGION_LABELS.get(region, region) for region in REGION_ORDER]

fig, ax = plt.subplots(figsize=(7.4, 5.2))
ax.boxplot(box_values, showmeans=True)
ax.set_xticks(range(1, len(box_labels) + 1))
ax.set_xticklabels(box_labels, rotation=20, ha="right")
ax.axhline(0, color="black", linewidth=1, linestyle="--")
ax.set_title("Stable Diffusion LPIPS improvement by region")
ax.set_ylabel("LPIPS improvement: damaged minus restored")
ax.grid(axis="y", alpha=0.25)

save_matplotlib_figure(
    fig,
    "stable_diffusion_lpips_improvement_boxplot_by_region.png",
    "aggregate_plot",
    "LPIPS improvement distribution by region",
    "Distribution of LPIPS improvement values across full image, content region, and mask-bbox crop.",
    source_table=rel(BATCH3_METRICS_PATH),
    source_rows=len(box_df),
)

if {"mask_type", "evaluation_region", "lpips_improvement"}.issubset(ok_batch5_metrics_df.columns):
    mask_summary_plot_df = (
        finite_plot_df(ok_batch5_metrics_df, ["lpips_improvement"])
        .groupby(["mask_type", "evaluation_region"], dropna=False)
        .agg(
            median_lpips_improvement=("lpips_improvement", "median"),
            rows=("lpips_improvement", "size"),
        )
        .reset_index()
    )
    mask_summary_plot_df["plot_label"] = (
        mask_summary_plot_df["mask_type"].astype(str)
        + " | "
        + mask_summary_plot_df["evaluation_region"].map(REGION_LABELS).fillna(mask_summary_plot_df["evaluation_region"].astype(str))
    )
    mask_summary_plot_df = mask_summary_plot_df.sort_values("median_lpips_improvement", ascending=True).tail(30)

    fig, ax = plt.subplots(figsize=(8.2, max(4.2, 0.28 * len(mask_summary_plot_df) + 1.4)))
    ax.barh(mask_summary_plot_df["plot_label"], mask_summary_plot_df["median_lpips_improvement"])
    ax.axvline(0, color="black", linewidth=1, linestyle="--")
    ax.set_title("Median LPIPS improvement by mask type and region")
    ax.set_xlabel("Median LPIPS improvement")
    ax.grid(axis="x", alpha=0.25)

    save_matplotlib_figure(
        fig,
        "stable_diffusion_lpips_median_improvement_by_mask_type_region.png",
        "aggregate_plot",
        "Median LPIPS improvement by mask type and region",
        "Compact ranking of mask-type and region groups by median LPIPS improvement.",
        source_table=rel(BATCH3_METRICS_PATH),
        source_rows=len(mask_summary_plot_df),
    )

mask_area_df = ok_batch5_metrics_df.loc[
    ok_batch5_metrics_df["evaluation_region"].astype(str).eq("mask_bbox_crop")
].copy()

if {"mask_area_pixels", "lpips_improvement"}.issubset(mask_area_df.columns):
    mask_area_df = finite_plot_df(mask_area_df, ["mask_area_pixels", "lpips_improvement"])
    if not mask_area_df.empty:
        fig, ax = plt.subplots(figsize=(7.4, 5.4))
        ax.scatter(mask_area_df["mask_area_pixels"], mask_area_df["lpips_improvement"], s=14, alpha=0.45)
        ax.axhline(0, color="black", linewidth=1, linestyle="--")
        ax.set_xscale("log")
        ax.set_title("Mask area versus mask-crop LPIPS improvement")
        ax.set_xlabel("Mask area pixels, log scale")
        ax.set_ylabel("LPIPS improvement")
        ax.grid(alpha=0.25)

        save_matplotlib_figure(
            fig,
            "stable_diffusion_lpips_mask_area_vs_improvement.png",
            "aggregate_plot",
            "Mask area versus LPIPS improvement",
            "Scatter plot checking whether larger masks relate to stronger or weaker local LPIPS improvement.",
            source_table=rel(BATCH3_METRICS_PATH),
            source_rows=len(mask_area_df),
        )

if {"prompt_variant_family", "evaluation_region", "lpips_improvement"}.issubset(ok_batch5_metrics_df.columns):
    prompt_plot_df = (
        finite_plot_df(ok_batch5_metrics_df, ["lpips_improvement"])
        .groupby(["prompt_variant_family", "evaluation_region"], dropna=False)
        .agg(
            median_lpips_improvement=("lpips_improvement", "median"),
            rows=("lpips_improvement", "size"),
        )
        .reset_index()
    )
    prompt_plot_df["plot_label"] = (
        prompt_plot_df["prompt_variant_family"].astype(str)
        + " | "
        + prompt_plot_df["evaluation_region"].map(REGION_LABELS).fillna(prompt_plot_df["evaluation_region"].astype(str))
    )
    prompt_plot_df = prompt_plot_df.sort_values("median_lpips_improvement", ascending=True).tail(30)

    if not prompt_plot_df.empty:
        fig, ax = plt.subplots(figsize=(8.2, max(4.2, 0.28 * len(prompt_plot_df) + 1.4)))
        ax.barh(prompt_plot_df["plot_label"], prompt_plot_df["median_lpips_improvement"])
        ax.axvline(0, color="black", linewidth=1, linestyle="--")
        ax.set_title("Median LPIPS improvement by prompt family and region")
        ax.set_xlabel("Median LPIPS improvement")
        ax.grid(axis="x", alpha=0.25)

        save_matplotlib_figure(
            fig,
            "stable_diffusion_lpips_median_improvement_by_prompt_family_region.png",
            "aggregate_plot",
            "Median LPIPS improvement by prompt family and region",
            "Compact prompt-family diagnostic plot for LPIPS improvement.",
            source_table=rel(BATCH3_METRICS_PATH),
            source_rows=len(prompt_plot_df),
        )

print(f"Aggregate figures so far: {len(figure_records)}")
display(pd.DataFrame(figure_records))

Aggregate figures so far: 5


,figure_id,figure_type,title,description,path,path_project_relative,exists,size_bytes,source_table,source_rows,created_at_utc
0,stable_diffusion_lpips_damaged_vs_restored_sca...,aggregate_plot,Damaged versus restored LPIPS,Points below the diagonal indicate lower LPIPS...,D:\Masters\FH\Thesis\painting-restoration-eval...,outputs/24_stable_diffusion_lpips_metrics/figu...,True,248235,outputs/24_stable_diffusion_lpips_metrics/metr...,2724,2026-08-08T07:49:12.133083+00:00
1,stable_diffusion_lpips_improvement_boxplot_by_...,aggregate_plot,LPIPS improvement distribution by region,Distribution of LPIPS improvement values acros...,D:\Masters\FH\Thesis\painting-restoration-eval...,outputs/24_stable_diffusion_lpips_metrics/figu...,True,56711,outputs/24_stable_diffusion_lpips_metrics/metr...,2724,2026-08-08T07:49:12.133083+00:00
2,stable_diffusion_lpips_median_improvement_by_m...,aggregate_plot,Median LPIPS improvement by mask type and region,Compact ranking of mask-type and region groups...,D:\Masters\FH\Thesis\painting-restoration-eval...,outputs/24_stable_diffusion_lpips_metrics/figu...,True,191489,outputs/24_stable_diffusion_lpips_metrics/metr...,30,2026-08-08T07:49:12.133083+00:00
3,stable_diffusion_lpips_mask_area_vs_improvement,aggregate_plot,Mask area versus LPIPS improvement,Scatter plot checking whether larger masks rel...,D:\Masters\FH\Thesis\painting-restoration-eval...,outputs/24_stable_diffusion_lpips_metrics/figu...,True,110143,outputs/24_stable_diffusion_lpips_metrics/metr...,834,2026-08-08T07:49:12.133083+00:00
4,stable_diffusion_lpips_median_improvement_by_p...,aggregate_plot,Median LPIPS improvement by prompt family and ...,Compact prompt-family diagnostic plot for LPIP...,D:\Masters\FH\Thesis\painting-restoration-eval...,outputs/24_stable_diffusion_lpips_metrics/figu...,True,57028,outputs/24_stable_diffusion_lpips_metrics/metr...,6,2026-08-08T07:49:12.133083+00:00


In [23]:
# Batch 5 / Cell 22 - Generate selected-case image panels and save figure manifest
def numeric_box_from_row(row: pd.Series) -> tuple[int, int, int, int] | None:
    required_columns = ["region_x_min", "region_y_min", "region_x_max", "region_y_max"]
    if not all(column in row.index for column in required_columns):
        return None

    try:
        box = tuple(int(float(row[column])) for column in required_columns)
    except Exception:
        return None

    left, upper, right, lower = box
    if right <= left or lower <= upper:
        return None

    return left, upper, right, lower


def load_panel_image(path_value: str | Path, crop_box: tuple[int, int, int, int] | None, force_mask: bool = False) -> Image.Image:
    image_path = resolve_image_path(path_value)
    if not image_path.is_file():
        raise FileNotFoundError(f"Panel image missing: {image_path}")

    with Image.open(image_path) as image:
        if force_mask:
            panel_image = image.convert("L").convert("RGB")
        else:
            panel_image = image.convert("RGB")

    if crop_box is not None:
        width, height = panel_image.size
        left, upper, right, lower = crop_box
        left = max(0, min(left, width))
        right = max(0, min(right, width))
        upper = max(0, min(upper, height))
        lower = max(0, min(lower, height))
        if right > left and lower > upper:
            panel_image = panel_image.crop((left, upper, right, lower))

    return ImageOps.contain(panel_image, (PANEL_THUMB_SIZE, PANEL_THUMB_SIZE))


def make_selected_case_panel(row: pd.Series) -> Path:
    selected_case_key = str(row.get("selected_case_key", row.get("lpips_row_id", "selected_case")))
    selection_group = str(row.get("selection_group", "selected"))
    evaluation_region = str(row.get("evaluation_region", "unknown_region"))
    candidate_id = str(row.get("candidate_id", row.get("metric_case_id", "")))

    crop_box = None if evaluation_region == "full_image" else numeric_box_from_row(row)

    panel_items = [
        ("Clean", row.get("clean_path", ""), False),
        ("Damaged", row.get("damaged_path", ""), False),
        ("Restored", row.get("restored_path", ""), False),
        ("Mask", row.get("mask_path", ""), True),
    ]

    thumb_w = PANEL_THUMB_SIZE
    thumb_h = PANEL_THUMB_SIZE
    label_h = 28
    title_h = 96
    padding = 16
    cols = len(panel_items)
    canvas_w = cols * thumb_w + (cols + 1) * padding
    canvas_h = title_h + thumb_h + label_h + 2 * padding

    canvas = Image.new("RGB", (canvas_w, canvas_h), "white")
    draw = ImageDraw.Draw(canvas)

    title_lines = [
        f"{selection_group}",
        f"{candidate_id} | {REGION_LABELS.get(evaluation_region, evaluation_region)}",
        f"damaged={row.get('damaged_lpips', np.nan):.4f} restored={row.get('restored_lpips', np.nan):.4f} improvement={row.get('lpips_improvement', np.nan):.4f}",
    ]

    y = 12
    for line in title_lines:
        draw.text((padding, y), line[:150], fill=(20, 20, 20))
        y += 24

    for item_index, (label, path_value, force_mask) in enumerate(panel_items):
        x = padding + item_index * (thumb_w + padding)
        y = title_h

        thumb = load_panel_image(path_value, crop_box=crop_box, force_mask=force_mask)
        thumb_canvas = Image.new("RGB", (thumb_w, thumb_h), (245, 245, 245))
        offset = ((thumb_w - thumb.width) // 2, (thumb_h - thumb.height) // 2)
        thumb_canvas.paste(thumb, offset)

        canvas.paste(thumb_canvas, (x, y))
        draw.rectangle((x, y, x + thumb_w, y + thumb_h), outline=(45, 45, 45), width=1)
        draw.text((x, y + thumb_h + 7), label, fill=(20, 20, 20))

    output_filename = f"{safe_slug(selection_group)}__{safe_slug(selected_case_key)}.png"
    output_path = BATCH5_FIGURE_DIR / output_filename
    canvas.save(output_path)

    register_figure(
        figure_id=output_path.stem,
        figure_type="selected_case_panel",
        title=f"Selected LPIPS case panel: {candidate_id}",
        description="Four-panel diagnostic view showing clean, damaged, restored, and mask images for the selected LPIPS row.",
        output_path=output_path,
        source_table=rel(BATCH4_SELECTED_CASES_PATH),
        source_rows=1,
        selected_case_key=selected_case_key,
        selection_group=selection_group,
        selection_rank=int(row.get("selection_rank", 0)) if pd.notna(row.get("selection_rank", np.nan)) else "",
        metric_case_id=str(row.get("metric_case_id", "")),
        lpips_row_id=str(row.get("lpips_row_id", "")),
        candidate_id=candidate_id,
        evaluation_region=evaluation_region,
        lpips_improvement=float(row.get("lpips_improvement", np.nan)),
    )

    return output_path


panel_input_df = (
    batch5_selected_cases_df
    .sort_values(["selection_group", "selection_rank", "selected_case_key"], kind="stable")
    .groupby("selection_group", dropna=False)
    .head(SELECTED_PANEL_ROWS_PER_GROUP)
    .reset_index(drop=True)
)

panel_paths = []
for _, selected_row in panel_input_df.iterrows():
    panel_paths.append(make_selected_case_panel(selected_row))

batch5_figure_manifest_df = pd.DataFrame(figure_records)

front_manifest_columns = [
    "figure_id",
    "figure_type",
    "title",
    "description",
    "selection_group",
    "selection_rank",
    "selected_case_key",
    "metric_case_id",
    "lpips_row_id",
    "candidate_id",
    "evaluation_region",
    "lpips_improvement",
    "path",
    "path_project_relative",
    "exists",
    "size_bytes",
    "source_table",
    "source_rows",
    "created_at_utc",
]
front_manifest_columns = [column for column in front_manifest_columns if column in batch5_figure_manifest_df.columns]
remaining_manifest_columns = [column for column in batch5_figure_manifest_df.columns if column not in front_manifest_columns]
batch5_figure_manifest_df = batch5_figure_manifest_df[front_manifest_columns + remaining_manifest_columns].copy()

batch5_figure_manifest_df.to_csv(BATCH5_FIGURE_MANIFEST_PATH, index=False)

print(f"Saved: {rel(BATCH5_FIGURE_MANIFEST_PATH)}")
print(f"Figure manifest rows: {len(batch5_figure_manifest_df):,}")
print(f"Selected-case panels: {len(panel_paths):,}")

display(batch5_figure_manifest_df)

Saved: outputs/24_stable_diffusion_lpips_metrics/figures/stable_diffusion_lpips_figure_manifest.csv
Figure manifest rows: 19
Selected-case panels: 14


,figure_id,figure_type,title,description,selection_group,selection_rank,selected_case_key,metric_case_id,lpips_row_id,candidate_id,evaluation_region,lpips_improvement,path,path_project_relative,exists,size_bytes,source_table,source_rows,created_at_utc
0,stable_diffusion_lpips_damaged_vs_restored_sca...,aggregate_plot,Damaged versus restored LPIPS,Points below the diagonal indicate lower LPIPS...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,D:\Masters\FH\Thesis\painting-restoration-eval...,outputs/24_stable_diffusion_lpips_metrics/figu...,True,248235,outputs/24_stable_diffusion_lpips_metrics/metr...,2724,2026-08-08T07:49:12.133083+00:00
1,stable_diffusion_lpips_improvement_boxplot_by_...,aggregate_plot,LPIPS improvement distribution by region,Distribution of LPIPS improvement values acros...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,D:\Masters\FH\Thesis\painting-restoration-eval...,outputs/24_stable_diffusion_lpips_metrics/figu...,True,56711,outputs/24_stable_diffusion_lpips_metrics/metr...,2724,2026-08-08T07:49:12.133083+00:00
2,stable_diffusion_lpips_median_improvement_by_m...,aggregate_plot,Median LPIPS improvement by mask type and region,Compact ranking of mask-type and region groups...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,D:\Masters\FH\Thesis\painting-restoration-eval...,outputs/24_stable_diffusion_lpips_metrics/figu...,True,191489,outputs/24_stable_diffusion_lpips_metrics/metr...,30,2026-08-08T07:49:12.133083+00:00
3,stable_diffusion_lpips_mask_area_vs_improvement,aggregate_plot,Mask area versus LPIPS improvement,Scatter plot checking whether larger masks rel...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,D:\Masters\FH\Thesis\painting-restoration-eval...,outputs/24_stable_diffusion_lpips_metrics/figu...,True,110143,outputs/24_stable_diffusion_lpips_metrics/metr...,834,2026-08-08T07:49:12.133083+00:00
4,stable_diffusion_lpips_median_improvement_by_p...,aggregate_plot,Median LPIPS improvement by prompt family and ...,Compact prompt-family diagnostic plot for LPIP...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,D:\Masters\FH\Thesis\painting-restoration-eval...,outputs/24_stable_diffusion_lpips_metrics/figu...,True,57028,outputs/24_stable_diffusion_lpips_metrics/metr...,6,2026-08-08T07:49:12.133083+00:00
5,strongest_improvement_content_region__stronges...,selected_case_panel,Selected LPIPS case panel: sd__can__p001__mixe...,"Four-panel diagnostic view showing clean, dama...",strongest_improvement__content_region,1.0,strongest_improvement__content_region__sd24_00...,sd24_0003,sd24_0003__content_region__lpips,sd__can__p001__mixed_damage__p00_generic__s202...,content_region,0.426079,D:\Masters\FH\Thesis\painting-restoration-eval...,outputs/24_stable_diffusion_lpips_metrics/figu...,True,143116,outputs/24_stable_diffusion_lpips_metrics/anal...,1,2026-08-08T07:49:12.133083+00:00
6,strongest_improvement_content_region__stronges...,selected_case_panel,Selected LPIPS case panel: sd__can__p022__mixe...,"Four-panel diagnostic view showing clean, dama...",strongest_improvement__content_region,2.0,strongest_improvement__content_region__sd24_01...,sd24_0188,sd24_0188__content_region__lpips,sd__can__p022__mixed_damage__p00_generic__s202...,content_region,0.373101,D:\Masters\FH\Thesis\painting-restoration-eval...,outputs/24_stable_diffusion_lpips_metrics/figu...,True,162422,outputs/24_stable_diffusion_lpips_metrics/anal...,1,2026-08-08T07:49:12.133083+00:00
7,strongest_improvement_full_image__strongest_im...,selected_case_panel,Selected LPIPS case panel: sd__can__p001__mixe...,"Four-panel diagnostic view showing clean, dama...",strongest_improvement__full_image,1.0,strongest_improvement__full_image__sd24_0003__...,sd24_0003,sd24_0003__full_image__lpips,sd__can__p001__mixed_damage__p00_generic__s202...,full_image,0.400647,D:\Masters\FH\Thesis\painting-restoration-eval...,outputs/24_stable_diffusion_lpips_metrics/figu...,True,143241,outputs/24_stable_diffusion_lpips_metrics/anal...,1,2026-08-08T07:49:12.133083+00:00
8,strongest_improvement_full_image__strongest_im...,selected_case_panel,Selected

In [24]:
# Batch 5 / Cell 23 - Validate diagnostic figures and update stage manifest
def batch5_validation_row(check_name: str, observed, expected, passed: bool, failure_message: str = "") -> dict[str, Any]:
    return {
        "check_name": check_name,
        "observed": observed,
        "expected": expected,
        "passed": bool(passed),
        "failure_message": "" if passed else failure_message,
    }


saved_figure_manifest_df = pd.read_csv(BATCH5_FIGURE_MANIFEST_PATH) if BATCH5_FIGURE_MANIFEST_PATH.is_file() else pd.DataFrame()

required_manifest_columns = [
    "figure_id",
    "figure_type",
    "title",
    "description",
    "path",
    "path_project_relative",
    "exists",
    "size_bytes",
    "source_table",
    "source_rows",
    "created_at_utc",
]
missing_manifest_columns = [
    column for column in required_manifest_columns
    if column not in saved_figure_manifest_df.columns
]

figure_path_exists = []
figure_size_positive = []

if not saved_figure_manifest_df.empty and "path" in saved_figure_manifest_df.columns:
    for path_value in saved_figure_manifest_df["path"]:
        figure_path = Path(str(path_value))
        figure_path_exists.append(figure_path.is_file())
        figure_size_positive.append(figure_path.is_file() and figure_path.stat().st_size > 0)

figure_type_counts = (
    saved_figure_manifest_df["figure_type"].value_counts(dropna=False).to_dict()
    if "figure_type" in saved_figure_manifest_df.columns
    else {}
)
figure_type_counts = {str(key): int(value) for key, value in figure_type_counts.items()}

duplicate_figure_ids = (
    int(saved_figure_manifest_df.duplicated(["figure_id"], keep=False).sum())
    if "figure_id" in saved_figure_manifest_df.columns
    else len(saved_figure_manifest_df)
)

selected_case_keys = (
    set(batch5_selected_cases_df["selected_case_key"].dropna().astype(str))
    if "selected_case_key" in batch5_selected_cases_df.columns
    else set()
)

panel_selected_keys = (
    set(
        saved_figure_manifest_df.loc[
            saved_figure_manifest_df["figure_type"].astype(str).eq("selected_case_panel"),
            "selected_case_key",
        ].dropna().astype(str)
    )
    if {"figure_type", "selected_case_key"}.issubset(saved_figure_manifest_df.columns)
    else set()
)

unknown_panel_keys = sorted(panel_selected_keys - selected_case_keys)

selected_groups = (
    set(batch5_selected_cases_df["selection_group"].dropna().astype(str))
    if "selection_group" in batch5_selected_cases_df.columns
    else set()
)

panel_groups = (
    set(
        saved_figure_manifest_df.loc[
            saved_figure_manifest_df["figure_type"].astype(str).eq("selected_case_panel"),
            "selection_group",
        ].dropna().astype(str)
    )
    if {"figure_type", "selection_group"}.issubset(saved_figure_manifest_df.columns)
    else set()
)

missing_panel_groups = sorted(selected_groups - panel_groups)

expected_min_aggregate_figures = 4
expected_min_selected_case_panels = min(
    len(selected_groups) * SELECTED_PANEL_ROWS_PER_GROUP,
    len(batch5_selected_cases_df),
)

batch5_validation_df = pd.DataFrame(
    [
        batch5_validation_row(
            "batch4_validation_passed",
            rel(BATCH4_VALIDATION_PATH),
            "all checks passed",
            BATCH4_VALIDATION_PATH.is_file()
            and bool_series(pd.read_csv(BATCH4_VALIDATION_PATH)["passed"]).all(),
            "Batch 4 validation is missing or did not pass.",
        ),
        batch5_validation_row(
            "figure_directory_exists",
            rel(BATCH5_FIGURE_DIR),
            "directory exists",
            BATCH5_FIGURE_DIR.is_dir(),
            "Batch 5 figure directory does not exist.",
        ),
        batch5_validation_row(
            "figure_manifest_written",
            rel(BATCH5_FIGURE_MANIFEST_PATH),
            "file exists",
            BATCH5_FIGURE_MANIFEST_PATH.is_file(),
            "Batch 5 figure manifest CSV was not written.",
        ),
        batch5_validation_row(
            "figure_manifest_non_empty",
            len(saved_figure_manifest_df),
            "> 0",
            len(saved_figure_manifest_df) > 0,
            "Batch 5 figure manifest is empty.",
        ),
        batch5_validation_row(
            "required_manifest_columns_present",
            missing_manifest_columns,
            [],
            len(missing_manifest_columns) == 0,
            "Figure manifest is missing required columns.",
        ),
        batch5_validation_row(
            "figure_ids_unique",
            duplicate_figure_ids,
            0,
            duplicate_figure_ids == 0,
            "Figure IDs are not unique.",
        ),
        batch5_validation_row(
            "aggregate_figures_created",
            figure_type_counts.get("aggregate_plot", 0),
            f">= {expected_min_aggregate_figures}",
            figure_type_counts.get("aggregate_plot", 0) >= expected_min_aggregate_figures,
            "Too few aggregate LPIPS diagnostic plots were created.",
        ),
        batch5_validation_row(
            "selected_case_panels_created",
            figure_type_counts.get("selected_case_panel", 0),
            f">= {expected_min_selected_case_panels}",
            figure_type_counts.get("selected_case_panel", 0) >= expected_min_selected_case_panels,
            "Too few selected-case image panels were created.",
        ),
        batch5_validation_row(
            "all_selected_groups_have_panel",
            missing_panel_groups[:20],
            [],
            len(missing_panel_groups) == 0,
            "One or more selected-case groups has no diagnostic panel.",
        ),
        batch5_validation_row(
            "selected_case_panels_map_to_batch4",
            unknown_panel_keys[:20],
            [],
            len(unknown_panel_keys) == 0,
            "One or more selected-case panel keys does not map back to Batch 4 selected cases.",
        ),
        batch5_validation_row(
            "all_manifest_paths_exist",
            int(sum(bool(value) for value in figure_path_exists)),
            len(saved_figure_manifest_df),
            len(figure_path_exists) == len(saved_figure_manifest_df) and all(figure_path_exists),
            "One or more figure paths in the manifest does not exist.",
        ),
        batch5_validation_row(
            "all_manifest_paths_nonempty",
            int(sum(bool(value) for value in figure_size_positive)),
            len(saved_figure_manifest_df),
            len(figure_size_positive) == len(saved_figure_manifest_df) and all(figure_size_positive),
            "One or more figure files is empty.",
        ),
    ]
)

batch5_validation_df.to_csv(BATCH5_VALIDATION_PATH, index=False)
batch5_passed = bool_series(batch5_validation_df["passed"]).all()

stage_manifest = json.loads(STAGE_MANIFEST_PATH.read_text(encoding="utf-8"))

stage_manifest["stage"] = "batch5_diagnostic_figures"
stage_manifest["stage_status"] = "passed" if batch5_passed else "failed"
stage_manifest["updated_at_utc"] = utc_now_iso()
stage_manifest["batch5"] = {
    "status": "passed" if batch5_passed else "failed",
    "figure_directory": rel(BATCH5_FIGURE_DIR),
    "figure_manifest": rel(BATCH5_FIGURE_MANIFEST_PATH),
    "validation": rel(BATCH5_VALIDATION_PATH),
    "figure_rows": int(len(saved_figure_manifest_df)),
    "figure_type_counts": figure_type_counts,
    "selected_panel_rows_per_group": int(SELECTED_PANEL_ROWS_PER_GROUP),
    "checks_passed": int(bool_series(batch5_validation_df["passed"]).sum()),
    "checks_total": int(len(batch5_validation_df)),
}

STAGE_MANIFEST_PATH.write_text(json.dumps(to_json_safe(stage_manifest), indent=2), encoding="utf-8")

print(f"Saved: {rel(BATCH5_VALIDATION_PATH)}")
print(f"Updated: {rel(STAGE_MANIFEST_PATH)}")
print(f"Batch 5 checks passed: {int(bool_series(batch5_validation_df['passed']).sum())} / {len(batch5_validation_df)}")

display(batch5_validation_df)

if not batch5_passed:
    display(batch5_validation_df.loc[~bool_series(batch5_validation_df["passed"]), ["check_name", "failure_message"]])
    raise RuntimeError("Batch 5 validation failed.")

Saved: outputs/24_stable_diffusion_lpips_metrics/validation/batch5_figure_validation.csv
Updated: outputs/24_stable_diffusion_lpips_metrics/manifests/stable_diffusion_lpips_metrics_stage_manifest.json
Batch 5 checks passed: 12 / 12


,check_name,observed,expected,passed,failure_message
0,batch4_validation_passed,outputs/24_stable_diffusion_lpips_metrics/vali...,all checks passed,True,
1,figure_directory_exists,outputs/24_stable_diffusion_lpips_metrics/figu...,directory exists,True,
2,figure_manifest_written,outputs/24_stable_diffusion_lpips_metrics/figu...,file exists,True,
3,figure_manifest_non_empty,19,> 0,True,
4,required_manifest_columns_present,[],[],True,
5,figure_ids_unique,0,0,True,
6,aggregate_figures_created,5,>= 4,True,
7,selected_case_panels_created,14,>= 14,True,
8,all_selected_groups_have_panel,[],[],True,
9,selected_case_panels_map_to_batch4,[],[],True,


In [25]:
# Batch 6 / Cell 24 - Load final inputs and define handoff helpers
import json
from pathlib import Path

BATCH6_ARTIFACT_INDEX_PATH = globals().get(
    "BATCH6_ARTIFACT_INDEX_PATH",
    OUTPUT_DIRS["manifests"] / "stable_diffusion_lpips_metrics_artifact_index.csv",
)
BATCH6_HANDOFF_MANIFEST_PATH = globals().get(
    "BATCH6_HANDOFF_MANIFEST_PATH",
    OUTPUT_DIRS["manifests"] / "stable_diffusion_lpips_metrics_handoff_manifest.json",
)
BATCH6_FINAL_VALIDATION_PATH = globals().get(
    "BATCH6_FINAL_VALIDATION_PATH",
    OUTPUT_DIRS["validation"] / "stable_diffusion_lpips_metrics_final_validation.csv",
)

BATCH5_FIGURE_MANIFEST_PATH = globals().get(
    "BATCH5_FIGURE_MANIFEST_PATH",
    OUTPUT_DIRS["figures"] / "stable_diffusion_lpips_figure_manifest.csv",
)
BATCH5_VALIDATION_PATH = globals().get(
    "BATCH5_VALIDATION_PATH",
    OUTPUT_DIRS["validation"] / "batch5_figure_validation.csv",
)

for output_dir in OUTPUT_DIRS.values():
    output_dir.mkdir(parents=True, exist_ok=True)

def batch6_read_csv_or_empty(path: Path) -> pd.DataFrame:
    return pd.read_csv(path) if Path(path).is_file() else pd.DataFrame()

def batch6_validation_file_passed(path: Path) -> bool:
    if not Path(path).is_file():
        return False
    df = pd.read_csv(path)
    return not df.empty and "passed" in df.columns and bool_series(df["passed"]).all()

def batch6_blank_path(value) -> bool:
    return pd.isna(value) or str(value).strip() == ""

def batch6_resolve_path(value) -> Path:
    if batch6_blank_path(value):
        return PROJECT_ROOT / "__missing_path__"
    return resolve_project_path(str(value).strip())

def batch6_path_columns(dataframe: pd.DataFrame) -> list[str]:
    return [
        column for column in dataframe.columns
        if (
            column == "path"
            or column.endswith("_path")
            or column.endswith("_file_path")
            or column.endswith("_filepath")
        )
    ]

batch6_input_cases_df = batch6_read_csv_or_empty(BATCH1_INPUT_CASES_PATH)
batch6_smoke_df = batch6_read_csv_or_empty(BATCH2_SMOKE_METRICS_PATH)
batch6_metrics_df = batch6_read_csv_or_empty(BATCH3_METRICS_PATH)
batch6_summary_df = batch6_read_csv_or_empty(BATCH4_SUMMARY_PATH)
batch6_selected_cases_df = batch6_read_csv_or_empty(BATCH4_SELECTED_CASES_PATH)
batch6_figure_manifest_df = batch6_read_csv_or_empty(BATCH5_FIGURE_MANIFEST_PATH)

print(f"Batch 6 metrics rows: {len(batch6_metrics_df):,}")
print(f"Batch 6 summary rows: {len(batch6_summary_df):,}")
print(f"Batch 6 selected rows: {len(batch6_selected_cases_df):,}")
print(f"Batch 6 figure manifest rows: {len(batch6_figure_manifest_df):,}")

Batch 6 metrics rows: 2,724
Batch 6 summary rows: 83
Batch 6 selected rows: 56
Batch 6 figure manifest rows: 19


In [26]:
# Batch 6 / Cell 25 - Build artifact index and write pending handoff manifest
def batch6_artifact_row(path_value, artifact_type: str, source_stage: str, artifact_id: str = "", expected_to_exist: bool = True) -> dict:
    artifact_path = batch6_resolve_path(path_value)
    exists = artifact_path.is_file()
    suffix = artifact_path.suffix.lower()

    row_count = np.nan
    json_key_count = np.nan

    if exists and suffix == ".csv":
        try:
            row_count = int(len(pd.read_csv(artifact_path)))
        except Exception:
            row_count = np.nan

    if exists and suffix == ".json":
        try:
            loaded_json = json.loads(artifact_path.read_text(encoding="utf-8"))
            json_key_count = int(len(loaded_json)) if isinstance(loaded_json, dict) else np.nan
        except Exception:
            json_key_count = np.nan

    return {
        "artifact_type": artifact_type,
        "source_stage": source_stage,
        "artifact_id": str(artifact_id),
        "artifact_path": rel(artifact_path),
        "filename": artifact_path.name,
        "extension": suffix,
        "exists": bool(exists),
        "file_size_bytes": int(artifact_path.stat().st_size) if exists else 0,
        "row_count": row_count,
        "json_key_count": json_key_count,
        "expected_to_exist": bool(expected_to_exist),
    }

def build_batch6_artifact_index_df() -> pd.DataFrame:
    rows = []

    known_artifacts = [
        (BATCH0_INVENTORY_SNAPSHOT_PATH, "batch0_inventory_snapshot_csv", "batch0", "batch0_inventory_snapshot"),
        (BATCH0_VALIDATION_PATH, "batch0_validation_csv", "batch0", "batch0_validation"),
        (BATCH1_INPUT_CASES_PATH, "batch1_input_cases_csv", "batch1", "batch1_input_cases"),
        (BATCH1_VALIDATION_PATH, "batch1_validation_csv", "batch1", "batch1_validation"),
        (BATCH2_SMOKE_METRICS_PATH, "batch2_smoke_metrics_csv", "batch2", "batch2_smoke_metrics"),
        (BATCH2_VALIDATION_PATH, "batch2_validation_csv", "batch2", "batch2_validation"),
        (BATCH3_METRICS_PATH, "batch3_lpips_metrics_csv", "batch3", "batch3_lpips_metrics"),
        (BATCH3_VALIDATION_PATH, "batch3_validation_csv", "batch3", "batch3_validation"),
        (BATCH4_SUMMARY_PATH, "batch4_summary_csv", "batch4", "batch4_summary"),
        (BATCH4_SELECTED_CASES_PATH, "batch4_selected_cases_csv", "batch4", "batch4_selected_cases"),
        (BATCH4_VALIDATION_PATH, "batch4_validation_csv", "batch4", "batch4_validation"),
        (BATCH5_FIGURE_MANIFEST_PATH, "batch5_figure_manifest_csv", "batch5", "batch5_figure_manifest"),
        (BATCH5_VALIDATION_PATH, "batch5_validation_csv", "batch5", "batch5_validation"),
        (BATCH6_ARTIFACT_INDEX_PATH, "batch6_artifact_index_csv", "batch6", "batch6_artifact_index"),
        (BATCH6_HANDOFF_MANIFEST_PATH, "batch6_handoff_manifest_json", "batch6", "batch6_handoff_manifest"),
        (BATCH6_FINAL_VALIDATION_PATH, "batch6_final_validation_csv", "batch6", "batch6_final_validation"),
        (STAGE_MANIFEST_PATH, "stage_manifest_json", "all_batches", "stage_manifest"),
    ]

    for path, artifact_type, source_stage, artifact_id in known_artifacts:
        rows.append(batch6_artifact_row(path, artifact_type, source_stage, artifact_id))

    if "path" in batch6_figure_manifest_df.columns:
        for _, row in batch6_figure_manifest_df.iterrows():
            rows.append(
                batch6_artifact_row(
                    row["path"],
                    f"batch5_figure__{row.get('figure_type', 'unknown')}",
                    "batch5",
                    row.get("figure_id", ""),
                )
            )

    selected_reference_columns = [
        column for column in ["clean_path", "damaged_path", "restored_path", "mask_path"]
        if column in batch6_selected_cases_df.columns
    ]
    for path_column in selected_reference_columns:
        for _, row in batch6_selected_cases_df.iterrows():
            rows.append(
                batch6_artifact_row(
                    row.get(path_column),
                    f"batch4_selected_reference__{path_column}",
                    "batch4",
                    row.get("selected_case_key", row.get("lpips_row_id", "")),
                )
            )

    artifact_index = pd.DataFrame(rows)
    artifact_index = artifact_index.drop_duplicates(
        ["artifact_type", "source_stage", "artifact_id", "artifact_path"],
        keep="first",
    )
    return artifact_index.sort_values(
        ["source_stage", "artifact_type", "artifact_id", "artifact_path"]
    ).reset_index(drop=True)

artifact_index_df = build_batch6_artifact_index_df()
artifact_index_df.to_csv(BATCH6_ARTIFACT_INDEX_PATH, index=False)

ok_metrics_df = (
    batch6_metrics_df.loc[batch6_metrics_df["status"].astype(str).eq("ok")].copy()
    if "status" in batch6_metrics_df.columns
    else batch6_metrics_df.copy()
)

observed_region_counts = (
    ok_metrics_df["evaluation_region"].astype(str).value_counts().sort_index().to_dict()
    if "evaluation_region" in ok_metrics_df.columns
    else {}
)
observed_region_counts = {str(key): int(value) for key, value in observed_region_counts.items()}

handoff_manifest = {
    "schema_version": "stable_diffusion_lpips_metrics_handoff_manifest_v1",
    "notebook_id": NOTEBOOK_ID,
    "notebook_label": NOTEBOOK_LABEL,
    "status": "pending_final_validation",
    "generated_at_utc": utc_now_iso(),
    "output_root": rel(OUTPUT_ROOT),
    "outputs": {
        "final_validation": rel(BATCH6_FINAL_VALIDATION_PATH),
        "artifact_index": rel(BATCH6_ARTIFACT_INDEX_PATH),
        "handoff_manifest": rel(BATCH6_HANDOFF_MANIFEST_PATH),
        "stage_manifest": rel(STAGE_MANIFEST_PATH),
        "input_cases": rel(BATCH1_INPUT_CASES_PATH),
        "lpips_metrics": rel(BATCH3_METRICS_PATH),
        "lpips_summary": rel(BATCH4_SUMMARY_PATH),
        "selected_cases": rel(BATCH4_SELECTED_CASES_PATH),
        "figure_manifest": rel(BATCH5_FIGURE_MANIFEST_PATH),
    },
    "counts": {
        "input_case_rows": int(len(batch6_input_cases_df)),
        "smoke_metric_rows": int(len(batch6_smoke_df)),
        "lpips_metric_rows": int(len(batch6_metrics_df)),
        "lpips_ok_metric_rows": int(len(ok_metrics_df)),
        "summary_rows": int(len(batch6_summary_df)),
        "selected_case_rows": int(len(batch6_selected_cases_df)),
        "figure_manifest_rows": int(len(batch6_figure_manifest_df)),
        "artifact_index_rows": int(len(artifact_index_df)),
        "region_counts": observed_region_counts,
    },
    "contracts": {
        "expected_candidate_rows": int(globals().get("EXPECTED_CANDIDATE_ROWS", len(batch6_input_cases_df))),
        "expected_lpips_rows": int(globals().get("EXPECTED_LPIPS_ROWS", len(batch6_metrics_df))),
        "expected_region_counts": globals().get("EXPECTED_REGION_COUNTS", {}),
        "lpips_net": globals().get("LPIPS_NET", ""),
        "lpips_input_size": int(globals().get("LPIPS_INPUT_SIZE", 0)),
        "region_policy": "full_image + content_region + mask_bbox_crop where mask bbox is applicable",
        "selected_cases_are_diagnostics": True,
        "figures_are_diagnostics": True,
    },
    "handoff_notes": {
        "metric_direction": "LPIPS improvement is damaged_lpips minus restored_lpips; higher is better.",
        "zero_control_policy": "Zero-control rows are retained for sanity checks and should not be interpreted as damaged-region restoration wins.",
        "downstream_use": "Use this manifest only after status is passed.",
    },
}

BATCH6_HANDOFF_MANIFEST_PATH.write_text(
    json.dumps(to_json_safe(handoff_manifest), indent=2),
    encoding="utf-8",
)

stage_manifest = json.loads(STAGE_MANIFEST_PATH.read_text(encoding="utf-8")) if STAGE_MANIFEST_PATH.is_file() else {
    "schema_version": "stable_diffusion_lpips_metrics_stage_manifest_v1",
    "notebook_id": NOTEBOOK_ID,
    "notebook_label": NOTEBOOK_LABEL,
    "output_root": rel(OUTPUT_ROOT),
}

stage_manifest["stage"] = "batch6_final_validation_artifact_index_handoff"
stage_manifest["stage_status"] = "pending_final_validation"
stage_manifest["updated_at_utc"] = utc_now_iso()
stage_manifest["batch6"] = {
    "status": "pending_final_validation",
    "outputs": handoff_manifest["outputs"],
    "counts": handoff_manifest["counts"],
    "contracts": handoff_manifest["contracts"],
}

STAGE_MANIFEST_PATH.write_text(json.dumps(to_json_safe(stage_manifest), indent=2), encoding="utf-8")

print(f"Saved: {rel(BATCH6_ARTIFACT_INDEX_PATH)}")
print(f"Saved: {rel(BATCH6_HANDOFF_MANIFEST_PATH)}")
print(f"Updated: {rel(STAGE_MANIFEST_PATH)}")
display(artifact_index_df.groupby(["source_stage", "artifact_type"], dropna=False).size().reset_index(name="artifact_count"))

Saved: outputs/24_stable_diffusion_lpips_metrics/manifests/stable_diffusion_lpips_metrics_artifact_index.csv
Saved: outputs/24_stable_diffusion_lpips_metrics/manifests/stable_diffusion_lpips_metrics_handoff_manifest.json
Updated: outputs/24_stable_diffusion_lpips_metrics/manifests/stable_diffusion_lpips_metrics_stage_manifest.json


,source_stage,artifact_type,artifact_count
0,all_batches,stage_manifest_json,1
1,batch0,batch0_inventory_snapshot_csv,1
2,batch0,batch0_validation_csv,1
3,batch1,batch1_input_cases_csv,1
4,batch1,batch1_validation_csv,1
5,batch2,batch2_smoke_metrics_csv,1
6,batch2,batch2_validation_csv,1
7,batch3,batch3_lpips_metrics_csv,1
8,batch3,batch3_validation_csv,1
9,batch4,batch4_selected_cases_csv,1


In [27]:
# Batch 6 / Cell 26 - Final validation and finalize handoff
artifact_index_df = build_batch6_artifact_index_df()
artifact_index_df.to_csv(BATCH6_ARTIFACT_INDEX_PATH, index=False)

required_metric_columns = [
    "metric_case_id",
    "evaluation_region",
    "damaged_lpips",
    "restored_lpips",
    "lpips_improvement",
]
missing_metric_columns = [column for column in required_metric_columns if column not in batch6_metrics_df.columns]

metric_numeric_finite = False
lpips_formula_consistent = False

if not missing_metric_columns:
    metric_numeric_df = ok_metrics_df[["damaged_lpips", "restored_lpips", "lpips_improvement"]].apply(pd.to_numeric, errors="coerce")
    metric_numeric_finite = bool(np.isfinite(metric_numeric_df.to_numpy(dtype=float)).all()) if not metric_numeric_df.empty else False
    if metric_numeric_finite:
        formula_delta = (
            metric_numeric_df["damaged_lpips"]
            - metric_numeric_df["restored_lpips"]
            - metric_numeric_df["lpips_improvement"]
        ).abs().max()
        lpips_formula_consistent = bool(formula_delta <= 1e-6)

expected_region_counts = globals().get("EXPECTED_REGION_COUNTS", {})
region_counts_match = (
    all(int(observed_region_counts.get(region, -1)) == int(expected_count) for region, expected_count in expected_region_counts.items())
    if expected_region_counts
    else bool(observed_region_counts)
)

metric_case_ids = set(batch6_metrics_df.get("metric_case_id", pd.Series(dtype=str)).dropna().astype(str))
selected_metric_case_ids = set(batch6_selected_cases_df.get("metric_case_id", pd.Series(dtype=str)).dropna().astype(str))
selected_cases_map_to_metrics = selected_metric_case_ids.issubset(metric_case_ids)

figure_paths = (
    batch6_figure_manifest_df["path"].dropna().astype(str).tolist()
    if "path" in batch6_figure_manifest_df.columns
    else []
)
figure_paths_existing = [
    batch6_resolve_path(path).is_file() and batch6_resolve_path(path).stat().st_size > 0
    for path in figure_paths
]

artifact_index_missing_non_final_df = artifact_index_df.loc[
    artifact_index_df["expected_to_exist"].astype(bool)
    & artifact_index_df["artifact_path"].ne(rel(BATCH6_FINAL_VALIDATION_PATH))
    & ~artifact_index_df["exists"].astype(bool)
].copy()

required_handoff_keys = {
    "schema_version",
    "notebook_id",
    "notebook_label",
    "status",
    "outputs",
    "counts",
    "contracts",
    "handoff_notes",
}
loaded_handoff_manifest = json.loads(BATCH6_HANDOFF_MANIFEST_PATH.read_text(encoding="utf-8")) if BATCH6_HANDOFF_MANIFEST_PATH.is_file() else {}

batch6_validation_rows = [
    validation_row("batch0_validation_passed", batch6_validation_file_passed(BATCH0_VALIDATION_PATH), True, batch6_validation_file_passed(BATCH0_VALIDATION_PATH), "Batch 0 validation did not pass."),
    validation_row("batch1_validation_passed", batch6_validation_file_passed(BATCH1_VALIDATION_PATH), True, batch6_validation_file_passed(BATCH1_VALIDATION_PATH), "Batch 1 validation did not pass."),
    validation_row("batch2_validation_passed", batch6_validation_file_passed(BATCH2_VALIDATION_PATH), True, batch6_validation_file_passed(BATCH2_VALIDATION_PATH), "Batch 2 validation did not pass."),
    validation_row("batch3_validation_passed", batch6_validation_file_passed(BATCH3_VALIDATION_PATH), True, batch6_validation_file_passed(BATCH3_VALIDATION_PATH), "Batch 3 validation did not pass."),
    validation_row("batch4_validation_passed", batch6_validation_file_passed(BATCH4_VALIDATION_PATH), True, batch6_validation_file_passed(BATCH4_VALIDATION_PATH), "Batch 4 validation did not pass."),
    validation_row("batch5_validation_passed", batch6_validation_file_passed(BATCH5_VALIDATION_PATH), True, batch6_validation_file_passed(BATCH5_VALIDATION_PATH), "Batch 5 validation did not pass."),
    validation_row("metrics_file_non_empty", len(batch6_metrics_df), "> 0", len(batch6_metrics_df) > 0, "LPIPS metrics CSV is empty or missing."),
    validation_row("metrics_required_columns_present", missing_metric_columns, [], len(missing_metric_columns) == 0, "LPIPS metrics CSV is missing required columns."),
    validation_row("metrics_row_count", len(batch6_metrics_df), globals().get("EXPECTED_LPIPS_ROWS", len(batch6_metrics_df)), len(batch6_metrics_df) == int(globals().get("EXPECTED_LPIPS_ROWS", len(batch6_metrics_df))), "LPIPS row count does not match the expected contract."),
    validation_row("region_counts_match_contract", observed_region_counts, expected_region_counts, region_counts_match, "LPIPS region counts do not match the expected contract."),
    validation_row("metric_numeric_values_finite", metric_numeric_finite, True, metric_numeric_finite, "One or more LPIPS numeric values is non-finite."),
    validation_row("lpips_improvement_formula_consistent", lpips_formula_consistent, True, lpips_formula_consistent, "lpips_improvement is not consistent with damaged_lpips minus restored_lpips."),
    validation_row("selected_cases_map_to_metrics", len(selected_metric_case_ids - metric_case_ids), 0, selected_cases_map_to_metrics, "One or more selected cases does not map back to Batch 3 metrics."),
    validation_row("figure_manifest_non_empty", len(batch6_figure_manifest_df), "> 0", len(batch6_figure_manifest_df) > 0, "Figure manifest is empty or missing."),
    validation_row("figure_files_exist", int(sum(figure_paths_existing)), len(figure_paths), len(figure_paths_existing) == len(figure_paths) and all(figure_paths_existing), "One or more figure files is missing or empty."),
    validation_row("artifact_index_missing_files", len(artifact_index_missing_non_final_df), 0, len(artifact_index_missing_non_final_df) == 0, "Artifact index contains missing expected files."),
    validation_row("handoff_manifest_required_keys_present", sorted(set(loaded_handoff_manifest.keys()).intersection(required_handoff_keys)), sorted(required_handoff_keys), required_handoff_keys.issubset(set(loaded_handoff_manifest.keys())), "Handoff manifest is missing required top-level keys."),
    validation_row("artifact_index_written", rel(BATCH6_ARTIFACT_INDEX_PATH), "file exists", BATCH6_ARTIFACT_INDEX_PATH.is_file(), "Artifact index CSV was not written."),
    validation_row("handoff_manifest_written", rel(BATCH6_HANDOFF_MANIFEST_PATH), "file exists", BATCH6_HANDOFF_MANIFEST_PATH.is_file(), "Handoff manifest JSON was not written."),
    validation_row("stage_manifest_written", rel(STAGE_MANIFEST_PATH), "file exists", STAGE_MANIFEST_PATH.is_file(), "Stage manifest JSON was not written."),
]

batch6_validation_df = pd.DataFrame(batch6_validation_rows)
batch6_validation_df.to_csv(BATCH6_FINAL_VALIDATION_PATH, index=False)

batch6_validation_df = pd.concat(
    [
        batch6_validation_df,
        pd.DataFrame([
            validation_row(
                "final_validation_written",
                rel(BATCH6_FINAL_VALIDATION_PATH),
                "file exists",
                BATCH6_FINAL_VALIDATION_PATH.is_file(),
                "Final validation CSV was not written.",
            )
        ]),
    ],
    ignore_index=True,
)
batch6_validation_df.to_csv(BATCH6_FINAL_VALIDATION_PATH, index=False)

batch6_passed = bool_series(batch6_validation_df["passed"]).all()

handoff_manifest = json.loads(BATCH6_HANDOFF_MANIFEST_PATH.read_text(encoding="utf-8"))
handoff_manifest["status"] = "passed" if batch6_passed else "failed"
handoff_manifest["finalized_at_utc"] = utc_now_iso()
handoff_manifest["final_validation"] = {
    "path": rel(BATCH6_FINAL_VALIDATION_PATH),
    "passed": bool(batch6_passed),
    "checks_passed": int(bool_series(batch6_validation_df["passed"]).sum()),
    "checks_total": int(len(batch6_validation_df)),
}
BATCH6_HANDOFF_MANIFEST_PATH.write_text(json.dumps(to_json_safe(handoff_manifest), indent=2), encoding="utf-8")

stage_manifest = json.loads(STAGE_MANIFEST_PATH.read_text(encoding="utf-8"))
stage_manifest["stage"] = "batch6_final_validation_artifact_index_handoff"
stage_manifest["stage_status"] = "passed" if batch6_passed else "failed"
stage_manifest["updated_at_utc"] = utc_now_iso()
stage_manifest["batch6"]["status"] = "passed" if batch6_passed else "failed"
stage_manifest["batch6"]["final_validation"] = handoff_manifest["final_validation"]
STAGE_MANIFEST_PATH.write_text(json.dumps(to_json_safe(stage_manifest), indent=2), encoding="utf-8")

artifact_index_df = build_batch6_artifact_index_df()
artifact_index_df.to_csv(BATCH6_ARTIFACT_INDEX_PATH, index=False)

print(f"Saved: {rel(BATCH6_FINAL_VALIDATION_PATH)}")
print(f"Finalized: {rel(BATCH6_HANDOFF_MANIFEST_PATH)}")
print(f"Refreshed: {rel(BATCH6_ARTIFACT_INDEX_PATH)}")
print(f"Batch 6 checks passed: {int(bool_series(batch6_validation_df['passed']).sum())} / {len(batch6_validation_df)}")

display(batch6_validation_df)

if not batch6_passed:
    display(batch6_validation_df.loc[~bool_series(batch6_validation_df["passed"]), ["check_name", "failure_message"]])
    raise RuntimeError("Batch 6 final validation failed.")

Saved: outputs/24_stable_diffusion_lpips_metrics/validation/stable_diffusion_lpips_metrics_final_validation.csv
Finalized: outputs/24_stable_diffusion_lpips_metrics/manifests/stable_diffusion_lpips_metrics_handoff_manifest.json
Refreshed: outputs/24_stable_diffusion_lpips_metrics/manifests/stable_diffusion_lpips_metrics_artifact_index.csv
Batch 6 checks passed: 21 / 21


,check_name,observed,expected,passed,failure_message
0,batch0_validation_passed,True,True,True,
1,batch1_validation_passed,True,True,True,
2,batch2_validation_passed,True,True,True,
3,batch3_validation_passed,True,True,True,
4,batch4_validation_passed,True,True,True,
5,batch5_validation_passed,True,True,True,
6,metrics_file_non_empty,2724,> 0,True,
7,metrics_required_columns_present,[],[],True,
8,metrics_row_count,2724,2724,True,
9,region_counts_match_contract,"{'content_region': 945, 'full_image': 945, 'ma...","{'full_image': 945, 'content_region': 945, 'ma...",True,
